Cell trajectories and morphology: reproducible thesis analysis

This notebook reads existing Cellpose exports without running segmentation again.
It separates detection quality, link quality, track summaries, and ML crop curation.
Original databases and images are never changed. Every exclusion is traceable.

Run all cells in order. All configuration is in Section 1. The track-level figures contain
one median per retained track. Section 7b additionally evaluates every usable single
detection, because one image is enough to measure the shape of a cell. Speeds and all other
trajectory quantities remain track-based.
A track is an analytical unit, not a guaranteed unique biological cell. Fragmented
tracks and overlapping clips can repeat cells. No p-values or biological-replicate
confidence intervals are inferred from thousands of frames.

The 600 uL/min condition is excluded throughout because of documented chip failure.
BF and Ph2 are shown separately by default: optical contrast can affect segmentation.
HeLa and NIH3T3 were recorded on different dates; cell type and recording day are
confounded in the supplied data. These are descriptive comparisons.

Crop decisions are conservative: partial cells and invalid images are excluded;
possible overlaps, source shape rejections and unstable segmentations are reviewed.
Unusual morphology is not itself a reason for rejection. Masks are not available,
so passing automatic checks is not proof of an isolated, correctly segmented cell.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from concurrent.futures import ThreadPoolExecutor
from importlib.metadata import version, PackageNotFoundError
import hashlib
import html
import io
import json
import os
import re
import sqlite3
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch, Rectangle
from matplotlib.ticker import MaxNLocator
import tifffile
from IPython.display import display

# Common figure style of the thesis (thesis_style.py in the same folder as this notebook)
import thesis_style as S

PIPELINE_VERSION = '2.2.0'
SOURCE_CACHE_VERSION = '2.0.0'  # source loading is unchanged; reuse verified raw-data caches

## 1. Configuration and declared assumptions

Flow is inferred only from the validated folder convention: `100_50_BF` means
100 + 50 = 150 uL/min through the imaged channel. CSV overrides take priority,
then stored database values, then folder inference. Conflicting flow values stop
execution instead of being silently pooled. Values and origins are exported.

Image calibration is stored in um/pixel, time in seconds and plotted speed in m/s.
PBS is assumed to be 1x without additives at 25 C (not a measured temperature).
Viscosity: 0.911 mPa s; density: an explicit approximation of 1000 kg/m3.
[PBS viscosity reference](https://www.ncbi.nlm.nih.gov/books/NBK604904/table/ncipr02.tab1/).
Local fluid velocity is not inferred from cell velocity. No Stokes force is reported.


In [ ]:
PROJECT_DIR = Path(r'D:\Code\zyla_ni_acquisition')
EXPERIMENTS = [
    dict(experiment_id='20260921_NIH3T3_single', enabled=True, cell_type='NIH3T3',
         database=r'D:\Luise\APD\210926\NIH3T3\phantom_event_clips_single\cellpose_flow_tracking_results\cell_tracking_results.sqlite',
         data_root=r'D:\Luise\APD\210926\NIH3T3\phantom_event_clips_single'),
    dict(experiment_id='20260922_HeLa_single', enabled=True, cell_type='HeLa',
         database=r'D:\Luise\APD\220926\HeLa\phantom_event_clips_single\cellpose_flow_tracking_results\cell_tracking_results.sqlite',
         data_root=r'D:\Luise\APD\220926\HeLa\phantom_event_clips_single'),
]
METADATA_CSV = PROJECT_DIR / 'cell_analysis_results' / 'cell_analysis_metadata.csv'
OUTPUT_ROOT = PROJECT_DIR / 'cell_thesis_results_v2_2'
REVIEW_CSV = OUTPUT_ROOT / 'manual_crop_review.csv'
# Optional environment override supports isolated validation runs.
OUTPUT_ROOT = Path(os.environ.get('CELL_THESIS_OUTPUT', str(OUTPUT_ROOT)))
REVIEW_CSV = OUTPUT_ROOT / 'manual_crop_review.csv'

PBS_TEMPERATURE_C = 25
PBS_VISCOSITY = {20: 1.023e-3, 25: 0.911e-3, 37: 0.713e-3}
PBS_DENSITY_KG_M3 = 1000.0
FLOW_DIRECTION_DEG = 180.0
EXCLUDED_FLOW_RATES = [600.0]
EDGE_MARGIN_PX = 3
MIN_SHARED_OBSERVATIONS = 3
MIN_TRACK_POINTS = 3
MIN_VALID_SEGMENTS = 2
REVERSE_TOLERANCE_PX = 1.0  # positional tolerance, not a fitted speed threshold
MAX_TRACK_AREA_CV = 0.35    # track identity warning; never a single-cell shape gate
AREA_SPIKE_RATIO = 1.8
ROBUST_Z_LIMIT = 6.0
# The source tracker used 50 px; retain that declared search constraint.
MAX_LINK_DISTANCE_PX = 50.0
MAX_SPEED_M_S = None       # optional independently justified physical bound
# Gap checks are explicit plausibility heuristics, not proof of cell identity.
MAX_MISSING_SAMPLED_DETECTIONS = 1
GAP_SPEED_RATIO_MIN = 0.5
GAP_SPEED_RATIO_MAX = 2.0
GAP_MAX_DIRECTION_CHANGE_DEG = 30.0
BBOX_OVERLAP_NOTE_THRESHOLD = 0.15  # informational only; does not reject a crop
CROP_WORKERS = min(32, os.cpu_count() or 1)
CROP_BATCH_SIZE = 1024
SHOW_FIGURES = True
COLORS = dict(S.CELL_COLORS)  # same cell colours as in the classification notebooks
# Single-detection morphology (Section 7b)
DETECTION_EXCLUDE_UNSTABLE_AREA_TRACKS = True  # drop detections of tracks with unstable area
AREA_DEFORMATION_METRIC = 'deformation_rtdc'   # y-axis of the area-deformation plot (RT-DC convention)
# Exposure time of the high-speed camera in microseconds, used for the motion-blur estimate.
# None: not known. A column exposure_us in the metadata CSV overrides this value per folder.
EXPOSURE_US = None
RNG = np.random.default_rng(42)

if not (isinstance(MAX_MISSING_SAMPLED_DETECTIONS, int) and MAX_MISSING_SAMPLED_DETECTIONS >= 0
        and 0 < GAP_SPEED_RATIO_MIN <= 1 <= GAP_SPEED_RATIO_MAX
        and 0 <= GAP_MAX_DIRECTION_CHANGE_DEG < 90
        and 0 <= BBOX_OVERLAP_NOTE_THRESHOLD <= 1):
    raise ValueError('Invalid gap-plausibility or overlap-note configuration.')
if PBS_TEMPERATURE_C not in PBS_VISCOSITY:
    raise ValueError('Use a documented PBS temperature or supply measured properties.')
RUN_DIR = OUTPUT_ROOT / ('run_' + datetime.now().strftime('%Y%m%d_%H%M%S_%f'))
TABLE_DIR, FIGURE_DIR = RUN_DIR / 'tables', RUN_DIR / 'figures'
CROP_DIR, CACHE_DIR = RUN_DIR / 'crop_curation', OUTPUT_ROOT / 'cache'
for directory in [TABLE_DIR, FIGURE_DIR, CROP_DIR, CACHE_DIR]:
    directory.mkdir(parents=True, exist_ok=True)
print('Run output:', RUN_DIR)

## 2. Read-only loading and provenance

Only videos listed as completed in the source database are used. Detection keys
`(experiment, video, frame, label)` must be unique. Track IDs are scoped to video
and experiment. Source morphology filters are preserved and reported; their bias
cannot be undone without inspecting or regenerating the original segmentation.
Input hashes include SQLite WAL sidecars. Local cache files are generated here;
never replace them with downloaded pickle files.


In [ ]:
KEYS = ['video', 'frame', 'label_id']
META_FIELDS = ['flow_rate_ul_min', 'fps', 'pixel_size_um_per_px',
               'channel_width_um', 'channel_height_um', 'viscosity_pa_s',
               'fluid_density_kg_m3', 'channel_center_y_px', 'exposure_us']

def stable_id(parts):
    return hashlib.sha256(json.dumps(list(parts), default=str).encode()).hexdigest()[:24]

def file_hash(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()

def read_table(conn, name, keep=None):
    cols = [r[1] for r in conn.execute(f'PRAGMA table_info("{name}")')]
    if not cols:
        return pd.DataFrame()
    selected = cols if keep is None else [c for c in keep if c in cols]
    query = ','.join('"' + c + '"' for c in selected)
    return pd.read_sql_query(f'SELECT {query} FROM "{name}"', conn)

def keyed(table, label):
    if table.empty:
        return table.set_index(KEYS) if set(KEYS).issubset(table) else pd.DataFrame()
    if table[KEYS].isna().any().any() or table.duplicated(KEYS).any():
        raise ValueError(f'Ambiguous detection keys in {label}')
    return table.set_index(KEYS)

def load_sources(specs):
    data, inventories, manifest = [], [], []
    for spec in specs:
        db = Path(spec['database']).resolve()
        if not db.is_file():
            raise FileNotFoundError(db)
        inputs = [p for p in [db, Path(str(db) + '-wal')] if p.exists()]
        before = [(str(p), p.stat().st_size, p.stat().st_mtime_ns) for p in inputs]
        digests = {str(p): file_hash(p) for p in inputs}
        cache_key = stable_id([SOURCE_CACHE_VERSION, spec, digests])
        cache_path = CACHE_DIR / f'load_{cache_key}.pkl'
        if cache_path.is_file():
            item = pd.read_pickle(cache_path)
            print('Cached:', spec['experiment_id'])
        else:
            with sqlite3.connect(db.as_uri() + '?mode=ro', uri=True) as conn:
                conn.execute('BEGIN')
                videos = read_table(conn, 'videos')
                detections = read_table(conn, 'detections')
                crop_columns = KEYS + ['track_uid', 'crop_status', 'crop_output_path',
                    'crop_margin_px', 'crop_final_height_px', 'crop_final_width_px']
                crops = read_table(conn, 'cell_crops', crop_columns)
                track_points = read_table(conn, 'track_points', KEYS + ['track_uid'])
                settings = read_table(conn, 'analysis_settings')
            if videos.empty or videos.video.duplicated().any():
                raise ValueError(f'Missing or duplicate video inventory: {db}')
            joined = keyed(detections[detections.video.isin(videos.video)], 'detections')
            for table, label in [(crops, 'cell_crops'), (track_points, 'track_points')]:
                if table.empty:
                    continue
                other = keyed(table[table.video.isin(videos.video)], label)
                common = joined.index.intersection(other.index)
                if 'track_uid' in joined and 'track_uid' in other:
                    a, b = joined.loc[common, 'track_uid'], other.loc[common, 'track_uid']
                    if (a.notna() & b.notna() & a.ne(b)).any():
                        raise ValueError('Conflicting source track IDs')
                joined = joined.combine_first(other)
            joined = joined.reset_index()
            video_meta = videos.set_index('video')
            for col in ['height_px', 'width_px', 'fps', 'pixel_size_um',
                        'flow_rate_ul_min', 'channel_width_um', 'channel_height_um',
                        'frame_step', 'fps_source', 'folder']:
                if col in video_meta:
                    values = joined.video.map(video_meta[col])
                    joined[col] = joined[col].combine_first(values) if col in joined else values
            item = (joined, videos, settings.to_dict('records'))
            pd.to_pickle(item, cache_path)
        if before != [(str(p), p.stat().st_size, p.stat().st_mtime_ns) for p in inputs]:
            raise RuntimeError('Source database changed during reading; retry after acquisition stops.')
        d, v, settings = item
        d, v = d.copy(), v.copy()
        d['experiment_id'] = spec['experiment_id']
        d['cell_type'] = spec['cell_type']
        d['source_video_path'] = d.video.map(lambda s: str(Path(spec['data_root']) / s))
        uid = d.get('track_uid', pd.Series(pd.NA, index=d.index)).astype('string')
        d['track_uid'] = uid.mask(uid.str.lower().isin(['', 'nan', 'none', '<na>']))
        d['detection_uid'] = [stable_id([spec['experiment_id'], v, int(f), int(l)])
                              for v, f, l in d[KEYS].itertuples(index=False, name=None)]
        d['cell_uid'] = [stable_id([spec['experiment_id'], v, 'track', t]) if pd.notna(t) else det
                        for v, t, det in d[['video', 'track_uid', 'detection_uid']].itertuples(index=False, name=None)]
        d['identity_kind'] = np.where(d.track_uid.notna(), 'tracked', 'unassigned')
        v['experiment_id'] = spec['experiment_id']
        v['loaded_detections'] = v.video.map(d.groupby('video').size()).fillna(0).astype(int)
        v['coverage_complete'] = v.loaded_detections.eq(v.get('n_segmented_objects', np.nan))
        data.append(d); inventories.append(v)
        manifest.append(dict(experiment_id=spec['experiment_id'], database=str(db),
                             hashes=digests, settings=settings, rows=len(d)))
    return pd.concat(data, ignore_index=True), pd.concat(inventories, ignore_index=True), manifest

enabled = [s for s in EXPERIMENTS if s.get('enabled', True)]
if not enabled or len({s['experiment_id'] for s in enabled}) != len(enabled):
    raise ValueError('Enable experiments with unique experiment_id values.')
if len({str(Path(s['database']).resolve()).casefold() for s in enabled}) != len(enabled):
    raise ValueError('Do not include a database twice.')
raw, video_inventory, source_manifest = load_sources(enabled)
assert raw.detection_uid.is_unique
print(f'Loaded {len(raw):,} detections from {len(video_inventory):,} completed clips.')
if not video_inventory.coverage_complete.all():
    warnings.warn('Some source exports are incomplete; counts cover saved detections only.')

## 3. Metadata resolution and exclusions known before analysis

Missing values never become zero. Nonempty CSV fields override stored values.
The folder parsing is anchored to the entire name, so unrelated digits cannot
become flow rates. BF/Ph2 remain separate acquisition modes. Physical properties
are assumptions unless explicitly overridden. Original frame numbers and FPS
define time; skipped frames retain their true elapsed time.


In [ ]:
metadata = pd.read_csv(METADATA_CSV) if METADATA_CSV.is_file() else pd.DataFrame()
points = raw.copy()
parsed = points['folder'].str.extract(r'^(?P<a>\d+(?:\.\d+)?)_(?P<b>\d+(?:\.\d+)?)_(?P<mode>BF|Ph2)$')
folder_flow = pd.to_numeric(parsed.a, errors='coerce') + pd.to_numeric(parsed.b, errors='coerce')
points['imaging_mode'] = parsed['mode'].fillna('unknown')
points['flow_rate_database'] = pd.to_numeric(points.get('flow_rate_ul_min', np.nan), errors='coerce')
points['flow_rate_folder'] = folder_flow
points['flow_rate_source'] = np.where(points.flow_rate_database.notna(), 'database', 'folder_sum')
points['flow_rate_ul_min'] = points.flow_rate_database.combine_first(folder_flow)
for col in ['fps', 'pixel_size_um', 'channel_width_um', 'channel_height_um']:
    points[col + '_source'] = np.where(points[col].notna(), 'database', 'missing')

if not metadata.empty:
    keys = ['experiment_id', 'folder']
    if not set(keys).issubset(metadata) or metadata.duplicated(keys).any():
        raise ValueError('CSV must have unique experiment_id/folder rows.')
    lookup = metadata.set_index(keys)
    index = pd.MultiIndex.from_frame(points[keys])
    for col in META_FIELDS:
        if col not in lookup:
            continue
        target = 'pixel_size_um' if col == 'pixel_size_um_per_px' else col
        original = lookup[col].reindex(index).reset_index(drop=True)
        values = pd.to_numeric(original, errors='coerce')
        if (original.notna() & values.isna()).any():
            raise ValueError(f'Non-numeric metadata: {col}')
        if target not in points:
            points[target] = np.nan
        points[target] = np.where(values.notna(), values, pd.to_numeric(points[target], errors='coerce'))
        source_col = 'flow_rate_source' if target == 'flow_rate_ul_min' else target + '_source'
        if source_col in points:
            points.loc[values.notna(), source_col] = 'metadata_csv'

conflict = points.flow_rate_folder.notna() & points.flow_rate_ul_min.notna() & ~np.isclose(
    points.flow_rate_folder, points.flow_rate_ul_min, rtol=0, atol=1e-6)
if conflict.any():
    display(points.loc[conflict, ['experiment_id', 'folder', 'flow_rate_folder', 'flow_rate_ul_min']].drop_duplicates())
    raise ValueError('Flow metadata conflicts with the confirmed folder convention.')
for col, default in [('viscosity_pa_s', PBS_VISCOSITY[PBS_TEMPERATURE_C]),
                     ('fluid_density_kg_m3', PBS_DENSITY_KG_M3)]:
    if col not in points:
        points[col] = np.nan
    points[col + '_source'] = np.where(points[col].notna(), 'metadata_or_database', 'PBS_assumption')
    points[col] = points[col].fillna(default)
points['fluid_temperature_c_assumed'] = PBS_TEMPERATURE_C
points['experiment_excluded'] = points.flow_rate_ul_min.isin(EXCLUDED_FLOW_RATES)
points['experiment_exclusion_reason'] = np.where(points.experiment_excluded, 'chip_failure_600_ul_min', '')
points['flow_group'] = points.flow_rate_ul_min.map(lambda x: f'{x:g} uL/min' if pd.notna(x) else 'unknown')
points['time_s'] = points.frame / points.fps.where(points.fps > 0)
points['x_um'] = points.centroid_x_px * points.pixel_size_um
points['y_um'] = points.centroid_y_px * points.pixel_size_um
points['area_um2'] = points.area * points.pixel_size_um**2
points['equivalent_diameter_um'] = 2 * np.sqrt(points.area_um2 / np.pi)
points['axis_ratio'] = points.major_axis_px / points.minor_axis_px
# D = (major - minor) / (major + minor), dimensionless.
points['deformation_recomputed'] = ((points.major_axis_px - points.minor_axis_px)
                                  / (points.major_axis_px + points.minor_axis_px))
points['bulk_velocity_m_s'] = points.flow_rate_ul_min * 1e3 / (60 * points.channel_width_um * points.channel_height_um)
points['hydraulic_diameter_m'] = 2 * points.channel_width_um * points.channel_height_um / (points.channel_width_um + points.channel_height_um) * 1e-6
points['channel_reynolds'] = points.fluid_density_kg_m3 * points.bulk_velocity_m_s * points.hydraulic_diameter_m / points.viscosity_pa_s
metadata_columns = ['experiment_id', 'folder', 'cell_type', 'imaging_mode', 'flow_rate_ul_min',
    'flow_rate_source', 'fps', 'fps_source', 'pixel_size_um', 'pixel_size_um_source',
    'channel_width_um', 'channel_width_um_source', 'channel_height_um', 'channel_height_um_source',
    'viscosity_pa_s', 'fluid_density_kg_m3', 'fluid_temperature_c_assumed', 'bulk_velocity_m_s',
    'channel_reynolds', 'experiment_exclusion_reason']
resolved_metadata = points[metadata_columns].drop_duplicates()
display(resolved_metadata)


## 4. Detection quality and auditable reasons

The main analysis requires source acceptance, verifiable image boundaries,
valid geometry and calibration. No percentile trimming of deformation, size,
circularity or elongation is applied. Circularity can slightly exceed one due
to discretized perimeter estimates; this alone is not an exclusion.
Within-track area spikes require both a factor >1.8 and a robust log-area
deviation >6 MAD-based standard deviations (minimum five observations).
Shape-related source rejections are kept in the review manifest, not called
confirmed bad cells. Primary analysis respects their original rejection.


In [ ]:
points['point_reasons'] = ''
def flag_points(mask, reason):
    mask = pd.Series(mask, index=points.index).fillna(False)
    points.loc[mask, 'point_reasons'] += reason + ';'

flag_points(points.experiment_excluded, 'chip_failure_600_ul_min')
flag_points(points.status.ne('accepted'), 'source_not_accepted')
geometry_cols = ['bbox_y0', 'bbox_x0', 'bbox_y1', 'bbox_x1', 'height_px', 'width_px']
points['border_verifiable'] = np.isfinite(points[geometry_cols]).all(axis=1)
points['at_image_edge'] = ((points.bbox_x0 <= EDGE_MARGIN_PX) | (points.bbox_y0 <= EDGE_MARGIN_PX)
    | (points.bbox_x1 >= points.width_px - EDGE_MARGIN_PX) | (points.bbox_y1 >= points.height_px - EDGE_MARGIN_PX))
flag_points(~points.border_verifiable, 'border_not_verifiable')
flag_points(points.at_image_edge, 'partial_or_edge_cell')
shape_cols = ['area', 'major_axis_px', 'minor_axis_px', 'deformation', 'solidity', 'circularity']
points['invalid_shape'] = (~np.isfinite(points[shape_cols]).all(axis=1)
    | (points.area <= 0) | (points.minor_axis_px <= 0)
    | (points.major_axis_px < points.minor_axis_px)
    | ~points.deformation.between(0, 1, inclusive='left')
    | ~points.solidity.between(0, 1) | (points.circularity <= 0)
    | (points.bbox_x1 <= points.bbox_x0) | (points.bbox_y1 <= points.bbox_y0))
flag_points(points.invalid_shape, 'invalid_measurements')
flag_points(~np.isfinite(points.pixel_size_um) | (points.pixel_size_um <= 0)
            | ~np.isfinite(points.fps) | (points.fps <= 0), 'invalid_calibration')
flag_points(points.flow_rate_ul_min.isna(), 'missing_flow_rate')
flag_points(~np.isclose(points.deformation, points.deformation_recomputed, atol=1e-5, equal_nan=False), 'inconsistent_deformation')

base = points.point_reasons.eq('') & points.identity_kind.eq('tracked')
subset = points.loc[base, ['cell_uid', 'area']].copy()
subset['log_area'] = np.log(subset.area)
groups = subset.groupby('cell_uid', sort=False)
center = groups.log_area.transform('median')
mad = subset.log_area.sub(center).abs().groupby(subset.cell_uid).transform('median')
count = groups.area.transform('size')
ratio = np.exp(subset.log_area.sub(center).abs())
spike = (count >= 5) & (ratio > AREA_SPIKE_RATIO) & (
    (subset.log_area.sub(center).abs() > ROBUST_Z_LIMIT * 1.4826 * mad) | (mad < 1e-12))
points['area_spike'] = False
points.loc[subset.index, 'area_spike'] = spike
flag_points(points.area_spike, 'within_track_area_spike')
points['point_qc_ok'] = points.point_reasons.eq('')
points['point_reasons'] = points.point_reasons.str.rstrip(';')
print('Detection QC:', int(points.point_qc_ok.sum()), '/', len(points))


## 5. Link audit and track retention

For each original link, dx and dy are in pixels, dt = delta(frame)/FPS,
and speed = hypot(dx,dy)*um_per_pixel/dt/1e6 in m/s. Direction uses image
coordinates (x right, y down); 180 degrees means right-to-left.

A gap may span **one missing sampled detection**. It passes only if both
endpoints pass point QC, the original search-distance and calibration checks
pass, and the immediately preceding AND following links are ordinary, valid,
forward-moving intervals. The gap must also move forward. Its speed must be
between 0.5 and 2 times EACH neighboring speed, and its displacement direction
must differ by at most 30 degrees from EACH neighbor. Longer, unsupported or
inconsistent gaps remain identity warnings and exclude the track. The
configuration exposes these pragmatic thresholds; they are not experimentally
validated identity guarantees. The gap speed is an interval average over the
actual elapsed time. No missing positions or shapes are interpolated, and no
new links are created.

`link_audit` records missing sample counts, both speed ratios, both angles,
support counts, acceptance, notes and rejection reasons. `track_audit` records
the number of accepted gaps. `gap_policy_counts` summarizes their effect.
The `without_bridged_gap_tracks` sensitivity scenario removes every retained
track that uses an accepted gap, showing the effect on the final population.

Reverse displacement beyond 1 pixel, duplicated frames, calibration changes,
links beyond the configured source search radius, and robust speed spikes are
still identity warnings. Speed-spike baselines use ordinary valid intervals;
gaps cannot make that reference more permissive. Any unresolved identity warning
excludes the whole original track. Invalid endpoints exclude their links.
At least three accepted detections and two usable links are required. Tracks
are never split into extra cells or automatically relinked. The source export
already trims track endpoints; a five-point sensitivity analysis quantifies the
selection effect of a stricter length minimum.

Area CV >0.35 is an identity warning, not proof of a switch. A sensitivity table
retains those tracks to expose the effect of this heuristic. An individual
unusual shape is not removed. Passing these checks does not prove track identity.

In [ ]:
tracked = points.loc[points.identity_kind.eq('tracked')].sort_values(['cell_uid', 'frame', 'label_id']).copy()
tg = tracked.groupby('cell_uid', sort=False)
previous = tg[['frame', 'time_s', 'centroid_x_px', 'centroid_y_px', 'point_qc_ok', 'detection_uid']].shift()
links = tracked[['cell_uid', 'detection_uid', 'experiment_id', 'cell_type', 'folder', 'imaging_mode',
                 'flow_rate_ul_min', 'video', 'frame', 'fps', 'pixel_size_um', 'frame_step']].copy()
links['from_detection_uid'] = previous.detection_uid
links['frame_delta'] = tracked.frame - previous.frame
links['dt_s'] = tracked.time_s - previous.time_s
links['dx_px'] = tracked.centroid_x_px - previous.centroid_x_px
links['dy_px'] = tracked.centroid_y_px - previous.centroid_y_px
links['step_px'] = np.hypot(links.dx_px, links.dy_px)
links['segment_time_s'] = (tracked.time_s + previous.time_s) / 2
links['speed_m_s'] = links.step_px * links.pixel_size_um / links.dt_s / 1e6
angle = np.deg2rad(FLOW_DIRECTION_DEG)
links['axial_step_px'] = links.dx_px * np.cos(angle) + links.dy_px * np.sin(angle)
links['axial_velocity_m_s'] = links.axial_step_px * links.pixel_size_um / links.dt_s / 1e6
links['endpoint_ok'] = tracked.point_qc_ok & previous.point_qc_ok.eq(True)
links = links.loc[links.from_detection_uid.notna()].copy()
links['link_reasons'] = ''
def flag_links(mask, reason):
    mask = pd.Series(mask, index=links.index).fillna(False)
    links.loc[mask, 'link_reasons'] += reason + ';'

flag_links(~links.endpoint_ok, 'invalid_endpoint')
flag_links(~np.isfinite(links.speed_m_s) | (links.dt_s <= 0), 'invalid_time_or_position')
# Sampling must be an integer multiple of the source's declared frame step.
sampled_steps = links.frame_delta / links.frame_step
links['gap_present'] = sampled_steps.gt(1)
links['missing_sampled_detections'] = (sampled_steps - 1).clip(lower=0)
sampling_valid = (np.isfinite(sampled_steps) & links.frame_step.gt(0)
                  & sampled_steps.ge(1) & np.isclose(sampled_steps, np.round(sampled_steps)))
flag_links(~sampling_valid, 'invalid_sampling_interval')
flag_links(links.axial_step_px < -REVERSE_TOLERANCE_PX, 'reverse_link')
flag_links(links.step_px > MAX_LINK_DISTANCE_PX, 'link_exceeds_search_distance')
if MAX_SPEED_M_S is not None:
    flag_links(links.speed_m_s > MAX_SPEED_M_S, 'physical_speed_limit')
calibration = tg[['fps', 'pixel_size_um', 'frame_step']].nunique(dropna=False).max(axis=1)
flag_links(links.cell_uid.map(calibration).gt(1), 'changing_calibration')
duplicate_ids = set(tracked.loc[tracked.duplicated(['cell_uid', 'frame'], keep=False), 'cell_uid'])
flag_links(links.cell_uid.isin(duplicate_ids), 'duplicate_track_frame')
valid_speed = links.speed_m_s.where(links.link_reasons.eq('') & ~links.gap_present)
median_speed = valid_speed.groupby(links.cell_uid).transform('median')
speed_mad = valid_speed.sub(median_speed).abs().groupby(links.cell_uid).transform('median')
n_speed = valid_speed.groupby(links.cell_uid).transform('count')
speed_spike = (n_speed >= 5) & (valid_speed > 2 * median_speed) & (
    valid_speed.sub(median_speed).abs() > ROBUST_Z_LIMIT * 1.4826 * speed_mad)
flag_links(speed_spike, 'speed_spike')

# Only immediately adjacent, ordinary intervals of the SAME track can support a gap.
# Invalid, reverse, stationary and spike intervals never supply a reference.
ordinary_support = (links.link_reasons.eq('') & ~links.gap_present
                    & links.speed_m_s.gt(0) & links.axial_step_px.gt(0))
reference_columns = ['speed_m_s', 'dx_px', 'dy_px', 'step_px']
reference = links[reference_columns].where(ordinary_support, axis=0)
reference['cell_uid'] = links.cell_uid
supported = []
speed_matches = []
direction_matches = []
for side, offset in [('before', 1), ('after', -1)]:
    neighbor = reference.groupby('cell_uid', sort=False)[reference_columns].shift(offset)
    has_support = neighbor.speed_m_s.notna()
    ratio = links.speed_m_s / neighbor.speed_m_s
    cosine = ((links.dx_px * neighbor.dx_px + links.dy_px * neighbor.dy_px)
              / (links.step_px * neighbor.step_px)).clip(-1, 1)
    direction_change = np.rad2deg(np.arccos(cosine))
    links[f'gap_speed_ratio_{side}'] = ratio.where(links.gap_present)
    links[f'gap_direction_change_{side}_deg'] = direction_change.where(links.gap_present)
    supported.append(has_support)
    speed_matches.append(ratio.between(GAP_SPEED_RATIO_MIN, GAP_SPEED_RATIO_MAX))
    direction_matches.append(direction_change.le(GAP_MAX_DIRECTION_CHANGE_DEG))
links['gap_support_count'] = (supported[0].astype(int) + supported[1].astype(int)).where(links.gap_present, 0)
both_supported = links.gap_support_count.eq(2)
short_gap = sampling_valid & links.missing_sampled_detections.le(MAX_MISSING_SAMPLED_DETECTIONS)
speed_consistent = speed_matches[0] & speed_matches[1]
direction_consistent = direction_matches[0] & direction_matches[1] & links.axial_step_px.gt(0)
links['gap_accepted'] = (links.gap_present & links.link_reasons.eq('') & short_gap
                         & both_supported & speed_consistent & direction_consistent)
flag_links(links.gap_present & ~short_gap, 'gap_exceeds_allowed_missing_steps')
flag_links(links.gap_present & ~both_supported, 'gap_without_neighbor_support')
flag_links(links.gap_present & both_supported & ~speed_consistent, 'gap_inconsistent_speed')
flag_links(links.gap_present & both_supported & ~direction_consistent, 'gap_inconsistent_direction')
flag_links(links.gap_present & ~links.gap_accepted, 'missing_intermediate_detection')
links['link_notes'] = np.where(links.gap_accepted, 'short_gap_supported_by_both_neighbors', '')
links['link_reasons'] = links.link_reasons.str.rstrip(';')
links['link_valid'] = links.link_reasons.eq('')
identity_terms = 'reverse_link|missing_intermediate_detection|link_exceeds_search_distance|physical_speed_limit|changing_calibration|duplicate_track_frame|speed_spike|invalid_time_or_position|invalid_sampling_interval'
links['identity_warning'] = links.link_reasons.str.contains(identity_terms, regex=True)

track_qc = tg.agg(experiment_id=('experiment_id', 'first'), cell_type=('cell_type', 'first'),
    flow_rate_ul_min=('flow_rate_ul_min', 'first'), folder=('folder', 'first'),
    imaging_mode=('imaging_mode', 'first'), video=('video', 'first'), track_uid=('track_uid', 'first'),
    n_detections=('detection_uid', 'size'), n_point_qc=('point_qc_ok', 'sum'),
    experiment_excluded=('experiment_excluded', 'any'))
valid_area = tracked.loc[tracked.point_qc_ok].groupby('cell_uid').area.agg(['mean', 'std', 'count'])
track_qc['area_cv'] = (valid_area['std'] / valid_area['mean']).where(valid_area['count'] >= 3)
link_groups = links.groupby('cell_uid')
track_qc['n_valid_segments'] = link_groups.link_valid.sum().reindex(track_qc.index, fill_value=0)
track_qc['n_suspect_links'] = link_groups.identity_warning.sum().reindex(track_qc.index, fill_value=0)
track_qc['n_bridged_gaps'] = link_groups.gap_accepted.sum().reindex(track_qc.index, fill_value=0)
track_qc['track_reasons'] = ''
for mask, reason in [
    (track_qc.experiment_excluded, 'chip_failure_600_ul_min'),
    (track_qc.n_point_qc < MIN_TRACK_POINTS, 'insufficient_valid_detections'),
    (track_qc.n_valid_segments < MIN_VALID_SEGMENTS, 'insufficient_valid_links'),
    (track_qc.n_suspect_links > 0, 'suspect_identity_links'),
    (track_qc.area_cv > MAX_TRACK_AREA_CV, 'unstable_area'),
]:
    track_qc.loc[mask, 'track_reasons'] += reason + ';'
track_qc['track_reasons'] = track_qc.track_reasons.str.rstrip(';')
track_qc['track_retained'] = track_qc.track_reasons.eq('')
track_qc['retained_without_area_cv_filter'] = track_qc.track_reasons.isin(['', 'unstable_area'])
track_qc = track_qc.reset_index()
status_map = track_qc.set_index('cell_uid')
points['track_retained'] = points.cell_uid.map(status_map.track_retained).eq(True)
points['track_reasons'] = points.cell_uid.map(status_map.track_reasons).fillna('unassigned_identity')
points['analysis_include'] = points.point_qc_ok & points.track_retained
links['used_in_primary'] = links.link_valid & links.cell_uid.map(status_map.track_retained).eq(True)
print('Retained tracks:', int(track_qc.track_retained.sum()), '/', len(track_qc))

## 6. Crop inspection, review queue and non-destructive curation

Every saved crop is decoded once and checked for file readability, finite pixels,
flat images and agreement with its declared dimensions. Features are cached in
SQLite by path, size and modification time; unchanged crops are not decoded again.
No image is resized, normalized or written back to the source.

Bounding-box overlap alone is weak evidence: rectangles can intersect even when
cell masks do not. Overlap above 0.15 of the smaller rectangle is recorded in
`crop_notes` as `bounding_boxes_overlap`; it does NOT trigger review or exclusion.
A second accepted detection center inside the crop still triggers review for a
possible additional cell. This is stronger evidence, but still not a confirmed
overlap. These checks cannot resolve overlapping cells that Cellpose merged into
one mask. Crops without flags are automatic candidates, not expert-certified labels.
No focus threshold is applied, as requested. Border clipping and invalid crops
are hard exclusions. Source shape rejections and area spikes are review cases.
Track rejection and crop rejection are separate: a good image can have a bad link.

`manual_crop_review.csv` accepts detection_uid, decision (`accept` or `reject`),
and comment. Manual acceptance resolves soft review flags only; it cannot override
corrupt images, chip failure or image-edge exclusions. Rerun from this section
after editing decisions. Original crops stay in place. Category folders contain
manifests referencing the originals and sampled contact sheets, avoiding hundreds
of thousands of duplicated files. A bounded export option appears in Section 11.

In [ ]:
# Neighbor evidence uses arrays to avoid repeated DataFrame writes per frame.
all_coords = points[['bbox_x0', 'bbox_y0', 'bbox_x1', 'bbox_y1']].to_numpy(float)
all_centers = points[['centroid_x_px', 'centroid_y_px']].to_numpy(float)
all_accepted = points.status.eq('accepted').to_numpy()
all_margins = points.crop_margin_px.fillna(3).to_numpy(float)
neighbor_flags = np.zeros(len(points), dtype=bool)
overlap_fractions = np.zeros(len(points), dtype=float)
frame_groups = points.groupby(['experiment_id', 'video', 'frame'], sort=False).indices
for idx in frame_groups.values():
    if len(idx) < 2:
        continue
    coords, centers = all_coords[idx], all_centers[idx]
    accepted_neighbor = all_accepted[idx]
    valid = np.isfinite(coords).all(axis=1)
    overlap_x = np.maximum(0, np.minimum(coords[:, None, 2], coords[None, :, 2]) - np.maximum(coords[:, None, 0], coords[None, :, 0]))
    overlap_y = np.maximum(0, np.minimum(coords[:, None, 3], coords[None, :, 3]) - np.maximum(coords[:, None, 1], coords[None, :, 1]))
    area = np.maximum(1, (coords[:, 2] - coords[:, 0]) * (coords[:, 3] - coords[:, 1]))
    overlap = overlap_x * overlap_y / np.minimum(area[:, None], area[None, :])
    overlap[:, ~accepted_neighbor] = 0
    np.fill_diagonal(overlap, 0)
    overlap_fractions[idx] = np.nan_to_num(overlap).max(axis=1)
    side = np.maximum(coords[:, 2] - coords[:, 0], coords[:, 3] - coords[:, 1]) + 2 * all_margins[idx]
    left = np.rint((coords[:, 0] + coords[:, 2]) / 2 - side / 2)
    top = np.rint((coords[:, 1] + coords[:, 3]) / 2 - side / 2)
    inside = ((centers[None, :, 0] >= left[:, None]) & (centers[None, :, 0] < (left + side)[:, None])
              & (centers[None, :, 1] >= top[:, None]) & (centers[None, :, 1] < (top + side)[:, None]))
    inside[:, ~accepted_neighbor] = False
    np.fill_diagonal(inside, False)
    neighbor_flags[idx] = inside.any(axis=1) & valid
points['neighbor_in_crop'] = neighbor_flags
points['bbox_overlap_fraction'] = overlap_fractions
print('Neighbor geometry audited.', flush=True)

FEATURE_VERSION = 1
feature_columns = ['path', 'size', 'mtime', 'version', 'height', 'width', 'pixel_range', 'sha256', 'error']
feature_db = CACHE_DIR / 'crop_features.sqlite'
with sqlite3.connect(feature_db) as cache:
    cache.execute('CREATE TABLE IF NOT EXISTS features (path TEXT PRIMARY KEY, size INTEGER, mtime INTEGER, version INTEGER, height INTEGER, width INTEGER, pixel_range REAL, sha256 TEXT, error TEXT)')
    existing = {row[0]: row for row in cache.execute('SELECT * FROM features')}

def inspect_crop(path):
    path = str(path)
    try:
        stat = Path(path).stat()
    except OSError as exc:
        return (path, -1, -1, FEATURE_VERSION, 0, 0, np.nan, '', type(exc).__name__)
    cached = existing.get(path)
    if cached and cached[1:4] == (stat.st_size, stat.st_mtime_ns, FEATURE_VERSION):
        return cached
    try:
        with Path(path).open('rb') as stream:
            content = stream.read()
        im = tifffile.imread(io.BytesIO(content))
        if im.ndim != 2 or im.size == 0:
            raise ValueError('non_2d_or_empty')
        if not np.isfinite(im).all():
            raise ValueError('nonfinite_pixels')
        digest = hashlib.sha256(str((im.shape, str(im.dtype))).encode() + im.tobytes()).hexdigest()
        return (path, stat.st_size, stat.st_mtime_ns, FEATURE_VERSION, im.shape[0], im.shape[1],
                float(im.max()) - float(im.min()), digest, '')
    except Exception as exc:
        return (path, stat.st_size, stat.st_mtime_ns, FEATURE_VERSION, 0, 0, np.nan, '',
                type(exc).__name__ + ':' + str(exc)[:100])

crop_paths = points.loc[points.crop_status.eq('saved') & points.crop_output_path.notna(), 'crop_output_path'].drop_duplicates().tolist()
feature_rows = []
started = time.perf_counter()
last_report = started
with ThreadPoolExecutor(max_workers=CROP_WORKERS) as executor, sqlite3.connect(feature_db) as cache:
    for offset in range(0, len(crop_paths), CROP_BATCH_SIZE):
        batch = list(executor.map(inspect_crop, crop_paths[offset:offset + CROP_BATCH_SIZE]))
        cache.executemany('INSERT OR REPLACE INTO features VALUES (?,?,?,?,?,?,?,?,?)', batch)
        cache.commit()
        feature_rows.extend(batch)
        if time.perf_counter() - last_report > 20 or offset + CROP_BATCH_SIZE >= len(crop_paths):
            print(f'Crop audit: {len(feature_rows):,}/{len(crop_paths):,}; {time.perf_counter()-started:.1f} s', flush=True)
            last_report = time.perf_counter()
features = pd.DataFrame(feature_rows, columns=feature_columns).rename(columns={'path': 'crop_output_path'})
crops = points[['detection_uid', 'cell_uid', 'track_uid', 'identity_kind', 'experiment_id', 'cell_type',
    'folder', 'video', 'frame', 'imaging_mode', 'flow_rate_ul_min', 'status', 'reason',
    'crop_output_path', 'crop_status', 'crop_final_height_px', 'crop_final_width_px',
    'point_reasons', 'track_reasons', 'track_retained', 'experiment_excluded', 'at_image_edge',
    'border_verifiable', 'invalid_shape', 'area_spike', 'neighbor_in_crop', 'bbox_overlap_fraction']].merge(
        features, on='crop_output_path', how='left', validate='many_to_one')
crops['crop_hard_reasons'] = ''
crops['crop_review_reasons'] = ''
crops['crop_notes'] = ''
def flag_crops(mask, reason, hard=False):
    col = 'crop_hard_reasons' if hard else 'crop_review_reasons'
    crops.loc[pd.Series(mask, index=crops.index).fillna(False), col] += reason + ';'

flag_crops(crops.experiment_excluded, 'chip_failure_600_ul_min', True)
flag_crops(crops.at_image_edge | crops.reason.fillna('').str.contains('partial cell', case=False), 'partial_or_edge_cell', True)
flag_crops(~crops.crop_status.eq('saved') | crops.crop_output_path.isna(), 'crop_not_saved', True)
flag_crops(crops.error.fillna('missing_crop').ne(''), 'unreadable_or_invalid_pixels', True)
flag_crops(crops.pixel_range.eq(0), 'flat_image', True)
flag_crops((crops.height.ne(crops.crop_final_height_px) | crops.width.ne(crops.crop_final_width_px)) & crops.error.eq(''), 'crop_dimension_mismatch', True)
flag_crops(~crops.border_verifiable, 'border_not_verifiable')
flag_crops(crops.invalid_shape, 'invalid_measurements')
flag_crops(crops.status.ne('accepted') & ~crops.reason.fillna('').str.contains('partial cell', case=False), 'source_segmentation_rejection')
flag_crops(crops.area_spike, 'within_track_area_spike')
flag_crops(crops.neighbor_in_crop, 'additional_cell_center_in_crop')
# A rectangle intersection is retained as evidence, not used as an exclusion.
crops.loc[crops.bbox_overlap_fraction.gt(BBOX_OVERLAP_NOTE_THRESHOLD), 'crop_notes'] = 'bounding_boxes_overlap'
for col in ['crop_hard_reasons', 'crop_review_reasons']:
    crops[col] = crops[col].str.rstrip(';')
crops['manual_decision'] = ''
crops['manual_comment'] = ''
if not REVIEW_CSV.exists():
    pd.DataFrame(columns=['detection_uid', 'decision', 'comment']).to_csv(REVIEW_CSV, index=False)
manual = pd.read_csv(REVIEW_CSV, dtype=str).fillna('')
if manual.detection_uid.duplicated().any() or not manual.decision.isin(['accept', 'reject', '']).all():
    raise ValueError('Manual decisions require unique IDs and accept/reject/blank decisions.')
if len(manual):
    decision_map = manual.set_index('detection_uid')
    crops['manual_decision'] = crops.detection_uid.map(decision_map.decision).fillna('')
    crops['manual_comment'] = crops.detection_uid.map(decision_map.comment).fillna('')
crops['crop_category'] = np.select([
    crops.crop_hard_reasons.ne('') | crops.manual_decision.eq('reject'),
    crops.manual_decision.eq('accept'),
    crops.crop_review_reasons.ne(''),
], ['excluded', 'accepted_reviewed', 'review'], default='accepted_candidate')
# Exact duplicates are documented, not silently counted as independent ML samples.
crops['image_duplicate_count'] = crops.sha256.where(crops.sha256.ne('')).map(
    crops.loc[crops.sha256.ne('') & crops.sha256.notna(), 'sha256'].value_counts()).fillna(0).astype(int)
crops['recording_group'] = crops.experiment_id + '|' + crops.folder
crops['ml_eligible'] = crops.crop_category.isin(['accepted_candidate', 'accepted_reviewed'])
# One image per known retained track; unassigned images stay in the full manifest.
ml = crops.loc[crops.ml_eligible & crops.track_retained].sort_values(['cell_uid', 'frame']).copy()
ml['_order'] = ml.groupby('cell_uid').cumcount()
ml['_mid_distance'] = (ml['_order'] - (ml.groupby('cell_uid').frame.transform('size') - 1) / 2).abs()
ml_representatives = ml.sort_values(['cell_uid', '_mid_distance', 'frame']).drop_duplicates('cell_uid').drop(columns=['_order', '_mid_distance'])
ml_representatives = ml_representatives.drop_duplicates('sha256')
crop_counts = crops.groupby(['experiment_id', 'imaging_mode', 'flow_rate_ul_min', 'crop_category'], dropna=False).size().rename('n_crops').reset_index()
display(crop_counts)
print('Pre-deduplication ML representative crop candidates:', len(ml_representatives))

## 6b. Overlapping clips and duplicate observations

The supplied clips overlap: identical crop pixels occur in multiple exported
events. The adjacent `events.csv` supplies each clip's original starting frame;
its peak frame and clip length are checked against the TIFF name and database.
Only confirmed mappings define absolute acquisition frames. Exact matches require
the same recording group, absolute frame, detection bounding box and pixel hash.

Two candidate tracks are considered overlapping copies only after at least three
distinct shared observations. Connected groups retain one original track, preferring
primary-QC passage and then the most valid links/detections. No tracks are stitched.
This removes confirmed repeated observations but cannot establish that all remaining
tracks are different biological cells. The pre-deduplication summary is exported.
Exact duplicate crop pixels are also kept out of the ML candidate pool, with a
`duplicate` manifest pointing to the preferred image. Original files stay intact.


In [ ]:
from collections import Counter
from itertools import combinations

points['source_recording'] = ''
points['absolute_source_frame'] = np.nan
clip_mapping_rows, event_sources = [], []
spec_by_id = {spec['experiment_id']: spec for spec in enabled}
for (exp, folder), group in points.groupby(['experiment_id', 'folder'], sort=False):
    folder_root = Path(spec_by_id[exp]['data_root']) / folder
    event_path, settings_path = folder_root / 'events.csv', folder_root / 'settings.json'
    recording = exp + '|' + str(folder)
    points.loc[group.index, 'source_recording'] = recording
    if not event_path.is_file():
        continue
    events = pd.read_csv(event_path)
    if not {'event_id', 'center_cine_frame', 'clip_start_frame', 'clip_frame_count'}.issubset(events) or events.event_id.duplicated().any():
        raise ValueError(f'Invalid source event table: {event_path}')
    settings = json.loads(settings_path.read_text(encoding='utf-8-sig')) if settings_path.is_file() else {}
    event_sources.append(dict(path=str(event_path), sha256=file_hash(event_path),
        settings_path=str(settings_path), settings=settings))
    event_lookup = events.set_index('event_id')
    vmeta = video_inventory.loc[video_inventory.experiment_id.eq(exp)].set_index('video')
    for video, vg in group.groupby('video', sort=False):
        match = re.match(r'^event_(\d+)_.*_frame_(\d+)\.tif$', Path(video).name, flags=re.I)
        mapped = False
        start = np.nan
        if match and int(match[1]) in event_lookup.index:
            event = event_lookup.loc[int(match[1])]
            if int(match[2]) == int(event.center_cine_frame) and int(event.clip_frame_count) == int(vmeta.loc[video, 'n_frames_total']):
                start = int(event.clip_start_frame)
                points.loc[vg.index, 'absolute_source_frame'] = vg.frame + start
                mapped = True
        clip_mapping_rows.append(dict(experiment_id=exp, video=video, source_recording=recording,
            cine_path=settings.get('cine_path', ''), clip_start_frame=start, absolute_mapping_verified=mapped))
clip_mapping = pd.DataFrame(clip_mapping_rows, columns=['experiment_id', 'video', 'source_recording', 'cine_path', 'clip_start_frame', 'absolute_mapping_verified'])
points['source_observation_uid'] = ''
image_hashes = crops.set_index('detection_uid').sha256
hash_values = points.detection_uid.map(image_hashes).fillna('')
valid_mapping = points.absolute_source_frame.notna() & hash_values.ne('')
cols = ['source_recording', 'absolute_source_frame', 'bbox_x0', 'bbox_y0', 'bbox_x1', 'bbox_y1']
obs = points.loc[valid_mapping, cols].copy()
obs['sha256'] = hash_values.loc[valid_mapping]
points.loc[valid_mapping, 'source_observation_uid'] = [stable_id(row) for row in obs.itertuples(index=False, name=None)]
track_qc['retained_pre_dedup'] = track_qc.track_retained
possible_ids = set(track_qc.loc[track_qc.retained_without_area_cv_filter, 'cell_uid'])
matches = points.loc[valid_mapping & points.cell_uid.isin(possible_ids), ['cell_uid', 'source_observation_uid']].drop_duplicates()
# Only observation IDs shared by different original tracks can support a duplicate.
matches = matches.loc[matches.source_observation_uid.duplicated(keep=False)]
pair_counts = Counter()
for _, group in matches.groupby('source_observation_uid', sort=False):
    pair_counts.update(combinations(sorted(group.cell_uid.unique()), 2))
parent = {uid: uid for uid in possible_ids}
def find_root(uid):
    while parent[uid] != uid:
        parent[uid] = parent[parent[uid]]
        uid = parent[uid]
    return uid
pair_rows = []
for (a, b), count in pair_counts.items():
    if count >= MIN_SHARED_OBSERVATIONS:
        ra, rb = find_root(a), find_root(b)
        if ra != rb:
            parent[max(ra, rb)] = min(ra, rb)
        pair_rows.append(dict(track_a=a, track_b=b, shared_observations=count))
duplicate_pairs = pd.DataFrame(pair_rows, columns=['track_a', 'track_b', 'shared_observations'])
track_qc['overlap_group'] = track_qc.cell_uid.map({uid: find_root(uid) for uid in possible_ids})
ranked = track_qc.loc[track_qc.cell_uid.isin(possible_ids)].sort_values(
    ['retained_pre_dedup', 'n_valid_segments', 'n_point_qc', 'area_cv', 'cell_uid'],
    ascending=[False, False, False, True, True])
canonical = ranked.drop_duplicates('overlap_group').set_index('overlap_group').cell_uid
track_qc['canonical_track_uid'] = track_qc.overlap_group.map(canonical)
track_qc['duplicate_overlap_track'] = track_qc.canonical_track_uid.notna() & track_qc.cell_uid.ne(track_qc.canonical_track_uid)
duplicate = track_qc.duplicate_overlap_track
track_qc.loc[duplicate, 'track_reasons'] = track_qc.loc[duplicate, 'track_reasons'].map(
    lambda s: (s + ';duplicate_overlapping_clip_track').strip(';'))
track_qc['track_retained'] &= ~duplicate
track_qc['retained_without_area_cv_filter'] &= ~duplicate
status_map = track_qc.set_index('cell_uid')
points['track_retained'] = points.cell_uid.map(status_map.track_retained).eq(True)
points['track_reasons'] = points.cell_uid.map(status_map.track_reasons).fillna('unassigned_identity')
points['analysis_include'] = points.point_qc_ok & points.track_retained
links['used_in_primary'] = links.link_valid & links.cell_uid.map(status_map.track_retained).eq(True)
crops['track_retained'] = crops.cell_uid.map(status_map.track_retained).eq(True)
crops['track_reasons'] = crops.cell_uid.map(status_map.track_reasons).fillna('unassigned_identity')
crops['source_observation_uid'] = crops.detection_uid.map(points.set_index('detection_uid').source_observation_uid)
crops['source_recording'] = crops.detection_uid.map(points.set_index('detection_uid').source_recording)
# Prefer an eligible, manually reviewed crop on a retained track for each pixel hash.
preferred_crops = crops.loc[crops.sha256.notna() & crops.sha256.ne('')].assign(
    _manual=crops.manual_decision.eq('accept')).sort_values(
        ['ml_eligible', '_manual', 'track_retained', 'detection_uid'], ascending=[False, False, False, True])
preferred_by_hash = preferred_crops.drop_duplicates('sha256').set_index('sha256').detection_uid
crops['duplicate_of_detection_uid'] = crops.sha256.map(preferred_by_hash)
is_copy = crops.duplicate_of_detection_uid.notna() & crops.detection_uid.ne(crops.duplicate_of_detection_uid)
crops.loc[~is_copy, 'duplicate_of_detection_uid'] = ''
crops['crop_category_before_dedup'] = crops.crop_category
crops.loc[is_copy & crops.ml_eligible, 'crop_category'] = 'duplicate'
crops['ml_eligible'] &= ~is_copy
# Select one eligible representative from each canonical retained track.
ml = crops.loc[crops.ml_eligible & crops.track_retained].sort_values(['cell_uid', 'frame']).copy()
ml['_order'] = ml.groupby('cell_uid').cumcount()
ml['_mid_distance'] = (ml['_order'] - (ml.groupby('cell_uid').frame.transform('size') - 1) / 2).abs()
ml_representatives = ml.sort_values(['cell_uid', '_mid_distance', 'frame']).drop_duplicates('cell_uid').drop(columns=['_order', '_mid_distance'])
assert ml_representatives.sha256.is_unique
crop_counts = crops.groupby(['experiment_id', 'imaging_mode', 'flow_rate_ul_min', 'crop_category'], dropna=False).size().rename('n_crops').reset_index()
print('Verified clip mappings:', int(clip_mapping.absolute_mapping_verified.sum()), '/', int(video_inventory.loaded_detections.gt(0).sum()), 'clips with saved detections')
print('Completed clips without saved detections:', int(video_inventory.loaded_detections.eq(0).sum()))
print('Primary tracks before / after overlap deduplication:', int(track_qc.retained_pre_dedup.sum()), '/', int(track_qc.track_retained.sum()))
print('Exact repeated crop files:', int(is_copy.sum()), '; representative candidates:', len(ml_representatives))


## 7. One summary per retained track and complete accounting

Morphology is the median of accepted detections within each original track.
Speed is the median of retained original intervals, in m/s. Different metrics
can have different valid sample counts. Counts are exported in tables instead
of cluttering the population figures. Stage losses are sequential and additive;
reason counts overlap and must not be summed. Detection, link, track and crop
counts are always kept in separate units.

Crop image curation is independent of trajectory QC: rejecting a link does not
imply rejecting a usable image. Primary morphology here follows detection/track
QC; crop-only review flags and informational notes are reported separately.
A second summary restricted to tracks with no crop-review/exclusion flags is
exported as an image-QC sensitivity analysis. A further sensitivity scenario
removes tracks containing accepted gaps. Informational notes are counted in
`qc_note_counts`, separately from exclusion reasons.

In [ ]:
MORPHOLOGY = ['deformation', 'area_um2', 'equivalent_diameter_um', 'axis_ratio', 'solidity', 'eccentricity', 'circularity']
GROUPS = ['experiment_id', 'cell_type', 'imaging_mode', 'flow_rate_ul_min']

def aggregate_tracks(accepted_ids):
    selected_points = points.loc[points.point_qc_ok & points.cell_uid.isin(accepted_ids)]
    by_track = selected_points.groupby('cell_uid', sort=False)
    values = by_track[MORPHOLOGY].median()
    counts = by_track[MORPHOLOGY].count().add_suffix('_n')
    identity = by_track[GROUPS + ['folder', 'video', 'track_uid']].first()
    out = identity.join(values).join(counts)
    out['n_detections'] = by_track.size()
    out['duration_ms'] = (by_track.time_s.max() - by_track.time_s.min()) * 1000
    kept_links = links.loc[links.link_valid & links.cell_uid.isin(accepted_ids)]
    ls = kept_links.groupby('cell_uid')
    out = out.join(ls[['speed_m_s', 'axial_velocity_m_s']].median())
    out['n_valid_segments'] = ls.size()
    out['short_track'] = out.n_detections.lt(5)
    out['speed_m_s_n'] = out.n_valid_segments.fillna(0).astype(int)
    out['bulk_velocity_m_s'] = by_track.bulk_velocity_m_s.first()
    out['cell_to_bulk_speed_ratio'] = out.speed_m_s / out.bulk_velocity_m_s
    return out.reset_index()

cells_pre_dedup = aggregate_tracks(set(track_qc.loc[track_qc.retained_pre_dedup, 'cell_uid']))
cells_primary = aggregate_tracks(set(track_qc.loc[track_qc.track_retained, 'cell_uid']))
cells_length_sensitivity = cells_primary.loc[(cells_primary.n_detections >= 5) & (cells_primary.n_valid_segments >= 3)].copy()
cells_area_sensitivity = aggregate_tracks(set(track_qc.loc[track_qc.retained_without_area_cv_filter, 'cell_uid']))
clean_crop_ids = set(crops.loc[~crops.ml_eligible, 'cell_uid'])
cells_image_sensitivity = cells_primary.loc[~cells_primary.cell_uid.isin(clean_crop_ids)].copy()
bridged_track_ids = set(track_qc.loc[track_qc.n_bridged_gaps.gt(0), 'cell_uid'])
cells_gap_sensitivity = cells_primary.loc[~cells_primary.cell_uid.isin(bridged_track_ids)].copy()

def summary_table(table, groups=GROUPS):
    metrics = MORPHOLOGY + ['speed_m_s', 'axial_velocity_m_s']
    long = table.melt(id_vars=groups + ['cell_uid'], value_vars=metrics, var_name='metric', value_name='value').dropna(subset=['value'])
    return long.groupby(groups + ['metric'], dropna=False).value.agg(
        n='count', mean='mean', median='median', q25=lambda s: s.quantile(.25), q75=lambda s: s.quantile(.75)
    ).reset_index()

summary = summary_table(cells_primary)
sensitivity = pd.concat([
    summary.assign(scenario='primary'),
    summary_table(cells_pre_dedup).assign(scenario='before_overlap_deduplication'),
    summary_table(cells_length_sensitivity).assign(scenario='minimum_five_detections'),
    summary_table(cells_area_sensitivity).assign(scenario='without_area_cv_filter'),
    summary_table(cells_image_sensitivity).assign(scenario='strict_crop_qc_tracks'),
    summary_table(cells_gap_sensitivity).assign(scenario='without_bridged_gap_tracks'),
], ignore_index=True)

stage_rows = []
def add_stages(table, groups, entity, stages):
    for key, part in table.groupby(groups, dropna=False, sort=True):
        key = key if isinstance(key, tuple) else (key,)
        active = pd.Series(True, index=part.index)
        previous = len(part)
        for order, (name, mask) in enumerate(stages):
            active &= mask.reindex(part.index).fillna(False).astype(bool)
            kept = int(active.sum())
            stage_rows.append(dict(zip(groups, key)) | dict(entity=entity, stage_order=order,
                stage=name, entering=previous, removed=previous-kept, remaining=kept))
            previous = kept

add_stages(points, GROUPS, 'detections', [
    ('loaded', pd.Series(True, index=points.index)),
    ('experiment_valid', ~points.experiment_excluded),
    ('source_accepted', points.status.eq('accepted')),
    ('interior_and_verifiable', points.border_verifiable & ~points.at_image_edge),
    ('detection_qc', points.point_qc_ok),
    ('assigned_to_track', points.identity_kind.eq('tracked')),
    ('retained_track', points.track_retained),
])
add_stages(track_qc, GROUPS, 'tracks', [
    ('loaded', pd.Series(True, index=track_qc.index)),
    ('experiment_valid', ~track_qc.experiment_excluded),
    ('enough_detections', track_qc.n_point_qc >= MIN_TRACK_POINTS),
    ('enough_links', track_qc.n_valid_segments >= MIN_VALID_SEGMENTS),
    ('identity_links_pass', track_qc.n_suspect_links.eq(0)),
    ('track_qc_before_deduplication', track_qc.retained_pre_dedup),
    ('deduplicated_primary', track_qc.track_retained),
])
add_stages(links, GROUPS, 'links', [
    ('original_links', pd.Series(True, index=links.index)),
    ('endpoints_pass', links.endpoint_ok),
    ('link_qc', links.link_valid),
    ('used_in_primary', links.used_in_primary),
])
accounting = pd.DataFrame(stage_rows)
assert (accounting.entering - accounting.removed == accounting.remaining).all()
assert len(cells_primary) == int(track_qc.track_retained.sum())
assert cells_primary.cell_uid.is_unique
assert not cells_primary.flow_rate_ul_min.isin(EXCLUDED_FLOW_RATES).any()

reason_tables = []
for table, entity, column in [
    (points, 'detections', 'point_reasons'), (links, 'links', 'link_reasons'),
    (track_qc, 'tracks', 'track_reasons'), (crops, 'crops_hard', 'crop_hard_reasons'),
    (crops, 'crops_review', 'crop_review_reasons'),
]:
    exploded = table[GROUPS + [column]].copy()
    exploded['reason'] = exploded[column].fillna('').str.split(';')
    exploded = exploded.explode('reason')
    counts = exploded.loc[exploded.reason.ne('')].groupby(GROUPS + ['reason'], dropna=False).size().rename('count').reset_index()
    reason_tables.append(counts.assign(entity=entity))
reason_counts = pd.concat(reason_tables, ignore_index=True)
# Notes are informational and must not be interpreted as additional exclusions.
note_tables = []
for table, entity, column in [(links, 'links', 'link_notes'), (crops, 'crops', 'crop_notes')]:
    noted = table.loc[table[column].ne(''), GROUPS + [column]].rename(columns={column: 'note'})
    note_tables.append(noted.groupby(GROUPS + ['note'], dropna=False).size().rename('count').reset_index().assign(entity=entity))
qc_note_counts = pd.concat(note_tables, ignore_index=True)
gap_policy_counts = links.loc[links.gap_present].assign(
    accepted_gap_in_primary=lambda d: d.gap_accepted & d.used_in_primary).groupby(GROUPS, dropna=False).agg(
    original_gap_links=('gap_present', 'size'), accepted_gap_links=('gap_accepted', 'sum'),
    accepted_gap_links_in_primary=('accepted_gap_in_primary', 'sum')).reset_index()
gap_track_counts = track_qc.loc[track_qc.track_retained & track_qc.n_bridged_gaps.gt(0)].groupby(GROUPS, dropna=False).size().rename('primary_tracks_with_gaps').reset_index()
gap_policy_counts = gap_policy_counts.merge(gap_track_counts, on=GROUPS, how='left')
gap_policy_counts['primary_tracks_with_gaps'] = gap_policy_counts.primary_tracks_with_gaps.fillna(0).astype(int)
# Count crops freed from a bounding-box-only review flag, including repeated files.
bbox_only_candidates = (crops.crop_notes.ne('') & crops.crop_review_reasons.eq('')
                        & crops.crop_hard_reasons.eq('') & crops.manual_decision.eq(''))
crop_policy_counts = crops.loc[bbox_only_candidates].groupby(GROUPS + ['crop_category'], dropna=False).size().rename('bbox_only_review_resolved').reset_index()
source_rejections = points.groupby(GROUPS + ['status', 'reason'], dropna=False).size().rename('count').reset_index()
considered = cells_primary.groupby(GROUPS, dropna=False).agg(
    n_tracks=('cell_uid', 'size'), n_tracks_with_speed=('speed_m_s', 'count'),
    n_detections=('n_detections', 'sum'), n_links=('n_valid_segments', 'sum')).reset_index()
considered = points[GROUPS].drop_duplicates().merge(considered, on=GROUPS, how='left')
for column in ['n_tracks', 'n_tracks_with_speed', 'n_detections', 'n_links']:
    considered[column] = considered[column].fillna(0).astype(int)
considered['condition_excluded'] = considered.flow_rate_ul_min.isin(EXCLUDED_FLOW_RATES)
unassigned_counts = points.loc[points.identity_kind.eq('unassigned')].groupby(GROUPS, dropna=False).agg(
    unassigned_loaded=('detection_uid', 'size'), unassigned_point_qc_pass=('point_qc_ok', 'sum')).reset_index()
display(considered)
print(f'PRIMARY: {len(cells_primary):,} tracks; {int(points.analysis_include.sum()):,} detections; {int(links.used_in_primary.sum()):,} links.')

## 7b. Single-detection morphology

Shape is a property of a single image: one frame of a cell inside the channel is enough to measure its
deformation. This section therefore evaluates every usable detection, including detections that were not
assigned to a track or whose track failed an identity check. Quantities that need a trajectory (speed,
velocity, track duration) remain track-based.

A detection is used only if
* it passes the detection QC of Section 4 (source acceptance, cell not touching the image edge, valid and
  consistent measurements, calibration and flow rate known, no within-track area spike),
* its crop passed the crop checks of Section 6 (readable, not flat, correct dimensions, no source
  segmentation rejection, no second accepted cell centre inside the crop, no manual rejection),
* it does not belong to a track flagged for unstable area (`DETECTION_EXCLUDE_UNSTABLE_AREA_TRACKS`),
  because an unstable area indicates segmentation changes within the track.

Overlapping clips repeat the same observation. Repeated observations are removed with the verified source
observation ID (recording, absolute frame, bounding box and pixel hash) and, where no mapping exists, with the
pixel hash of the crop. A detection is an image of a cell, not a unique cell: unassigned detections of the same
cell in consecutive frames can still occur several times. Results are therefore reported in detections, and a
cell-weighted scenario (one median per retained track plus every unassigned detection) is given for comparison.

Additional quantities per detection:
* `deformation_rtdc` = 1 − c, the deformation parameter of real-time deformability cytometry, computed from the
  stored circularity c = 4πA/P² (Crofton perimeter). It can be slightly negative because the discrete perimeter
  estimate allows c > 1.
* `inclination_deg`: angle between the major axis of the fitted ellipse and the flow axis, 0° = aligned with the
  flow, 90° = perpendicular. For almost round masks the angle is poorly defined.
* `flow_aligned_deformation` = D cos(2 · inclination). Elongation along the flow gives positive values,
  perpendicular elongation negative values. Segmentation noise without a preferred direction averages towards
  zero, whereas a systematic elongation along the flow does not.
* `confinement_ratio` χ = d_eq / H, equivalent diameter over channel height.
* `motion_blur_um` = speed × exposure time, only if the exposure time is supplied (`EXPOSURE_US` or an
  `exposure_us` column in the metadata CSV). Blur stretches the image along the flow and increases D.

The within-track variation of D between consecutive frames of the same retained track estimates how strongly
segmentation noise contributes to single-frame values.

In [ ]:
# ---------------------------------------------------------------- per-detection derived quantities
points['deformation_rtdc'] = 1 - points.circularity
points['confinement_ratio'] = points.equivalent_diameter_um / points.channel_height_um
HAS_ORIENTATION = 'orientation_rad' in points and points.orientation_rad.notna().any()
if HAS_ORIENTATION:
    # skimage: orientation = angle between the image row axis and the major axis.
    major_axis_vs_x_deg = 90.0 - np.rad2deg(points.orientation_rad.astype(float))
    offset = (major_axis_vs_x_deg - (FLOW_DIRECTION_DEG % 180.0)) % 180.0
    points['inclination_deg'] = np.minimum(offset, 180.0 - offset)
    points['flow_aligned_deformation'] = points.deformation * np.cos(2 * np.deg2rad(points.inclination_deg))
else:
    points['inclination_deg'] = np.nan
    points['flow_aligned_deformation'] = np.nan
    print('No orientation stored in the source database: inclination and flow-aligned deformation are skipped.')

# ---------------------------------------------------------------- detection selection
crop_info = crops.set_index('detection_uid')
crop_ok = points.detection_uid.map(crop_info.crop_category_before_dedup).isin(['accepted_candidate', 'accepted_reviewed'])
unstable_track = points.track_reasons.fillna('').str.contains('unstable_area')
points['detection_analysis_include'] = (points.point_qc_ok & ~points.experiment_excluded & crop_ok
                                        & ~(unstable_track & DETECTION_EXCLUDE_UNSTABLE_AREA_TRACKS))
selected = points.loc[points.detection_analysis_include].copy()
pixel_hash = selected.detection_uid.map(crop_info.sha256).fillna('')
dedup_key = selected.source_observation_uid.where(selected.source_observation_uid.ne(''), 'sha:' + pixel_hash)
dedup_key = dedup_key.where(dedup_key.ne('sha:'), 'det:' + selected.detection_uid)
# Prefer the copy that belongs to a retained track, then a deterministic order.
selected = (selected.assign(_key=dedup_key, _not_retained=~selected.track_retained)
            .sort_values(['_not_retained', 'detection_uid']).drop_duplicates('_key')
            .drop(columns=['_key', '_not_retained']))
points['detection_dedup_keep'] = points.index.isin(selected.index)

# Speed is a track property: retained tracks use their own median speed, all other detections the median
# track speed of their group. It is only used for the optional motion-blur estimate.
track_speed = selected.cell_uid.map(cells_primary.set_index('cell_uid').speed_m_s)
group_speed = cells_primary.groupby(GROUPS).speed_m_s.median()
fallback = pd.Series([group_speed.get(tuple(k), np.nan) for k in selected[GROUPS].itertuples(index=False, name=None)],
                     index=selected.index)
selected['speed_estimate_m_s'] = track_speed.fillna(fallback)
selected['speed_source'] = np.where(track_speed.notna(), 'own_track', 'group_median_track_speed')
if 'exposure_us' not in points:
    points['exposure_us'] = np.nan
exposure = points.loc[selected.index, 'exposure_us']
if EXPOSURE_US is not None:
    exposure = exposure.fillna(float(EXPOSURE_US))
selected['exposure_us'] = exposure
selected['motion_blur_um'] = selected.speed_estimate_m_s * selected.exposure_us   # (m/s) * us = um
selected['motion_blur_fraction'] = selected.motion_blur_um / selected.equivalent_diameter_um
HAS_EXPOSURE = selected.exposure_us.notna().any()
if not HAS_EXPOSURE:
    print('No exposure time supplied: the motion-blur estimate is skipped (set EXPOSURE_US or add exposure_us to the metadata CSV).')

DETECTION_METRICS = ['deformation', 'deformation_rtdc', 'axis_ratio', 'area_um2', 'equivalent_diameter_um',
                     'circularity', 'solidity', 'confinement_ratio']
DETECTION_METRICS += [m for m in ['radial_std_norm', 'convexity_ratio'] if m in selected]
if HAS_ORIENTATION:
    DETECTION_METRICS += ['inclination_deg', 'flow_aligned_deformation']
if HAS_EXPOSURE:
    DETECTION_METRICS += ['motion_blur_um', 'motion_blur_fraction']

def metric_summary(table, metrics, groups=GROUPS):
    long = table.melt(id_vars=groups, value_vars=[m for m in metrics if m in table],
                      var_name='metric', value_name='value').dropna(subset=['value'])
    return long.groupby(groups + ['metric'], dropna=False).value.agg(
        n='count', mean='mean', median='median', q25=lambda s: s.quantile(.25), q75=lambda s: s.quantile(.75)
    ).reset_index()

# Track medians of the new quantities, so that track and detection level can be compared directly.
extra = [m for m in DETECTION_METRICS if m not in MORPHOLOGY and m in points and not m.startswith('motion_blur')]
track_extra = points.loc[points.analysis_include].groupby('cell_uid')[extra].median()
cells_primary = cells_primary.drop(columns=[c for c in extra if c in cells_primary]).merge(
    track_extra, left_on='cell_uid', right_index=True, how='left')

unassigned_selected = selected.loc[selected.identity_kind.eq('unassigned')]
cell_weighted = pd.concat([cells_primary[GROUPS + ['cell_uid'] + [m for m in DETECTION_METRICS if m in cells_primary]],
                           unassigned_selected[GROUPS + ['cell_uid'] + [m for m in DETECTION_METRICS if m in unassigned_selected]]],
                          ignore_index=True)
detection_summary = pd.concat([
    metric_summary(cells_primary, DETECTION_METRICS).assign(scenario='track_medians_primary', unit='tracks'),
    metric_summary(selected, DETECTION_METRICS).assign(scenario='all_detections', unit='detections'),
    metric_summary(selected.loc[selected.identity_kind.eq('tracked')], DETECTION_METRICS).assign(scenario='detections_on_tracks', unit='detections'),
    metric_summary(unassigned_selected, DETECTION_METRICS).assign(scenario='unassigned_detections', unit='detections'),
    metric_summary(cell_weighted, DETECTION_METRICS).assign(scenario='cell_weighted', unit='tracks + unassigned detections'),
], ignore_index=True)

# Counts of every selection step, in detections.
detection_counts = points.loc[~points.experiment_excluded].assign(
    crop_ok=crop_ok, unstable_track=unstable_track).groupby(GROUPS, dropna=False).agg(
    loaded=('detection_uid', 'size'), detection_qc=('point_qc_ok', 'sum'),
    crop_qc=('crop_ok', 'sum'), selected_before_dedup=('detection_analysis_include', 'sum'),
    selected_after_dedup=('detection_dedup_keep', 'sum')).reset_index()
detection_counts = detection_counts.merge(
    selected.groupby(GROUPS).identity_kind.value_counts().unstack(fill_value=0)
    .rename(columns={'tracked': 'of_which_on_tracks', 'unassigned': 'of_which_unassigned'}).reset_index(),
    on=GROUPS, how='left')
display(detection_counts)

# ---------------------------------------------------------------- repeatability within tracks
REPEATABILITY_MIN_POINTS = 5
rep_points = points.loc[points.analysis_include]
rep = rep_points.groupby('cell_uid').agg(n=('deformation', 'count'), d_sd=('deformation', 'std'),
                                          d_rtdc_sd=('deformation_rtdc', 'std'))
rep = rep.loc[rep.n >= REPEATABILITY_MIN_POINTS].join(cells_primary.set_index('cell_uid')[GROUPS + ['deformation', 'deformation_rtdc']])
track_repeatability = rep.groupby(GROUPS).agg(
    n_tracks=('n', 'size'), within_track_sd_D=('d_sd', 'median'),
    between_track_sd_D=('deformation', 'std'), within_track_sd_D_rtdc=('d_rtdc_sd', 'median'),
    between_track_sd_D_rtdc=('deformation_rtdc', 'std')).reset_index()
track_repeatability['within_to_between_D'] = track_repeatability.within_track_sd_D / track_repeatability.between_track_sd_D
display(track_repeatability.round(4))

key_view = detection_summary.loc[detection_summary.metric.isin(['deformation', 'deformation_rtdc'])
                                 & detection_summary.scenario.isin(['track_medians_primary', 'all_detections'])]
display(key_view.pivot_table(index=['imaging_mode', 'flow_rate_ul_min', 'cell_type'], columns=['metric', 'scenario'],
                             values='median').round(3))
print(f'Single-detection analysis: {len(selected):,} detections '
      f'({int(selected.identity_kind.eq("unassigned").sum()):,} without track).')

## 8. Publication figures and captions

Flow groups are ordered numerically. All population plots are stratified by optical
mode. Blue/orange identify cell types consistently. Speeds are in m/s.
Violin densities use all valid track medians and are normalized independently;
width is not sample size. Boxes are median/IQR with 1.5-IQR whiskers. Groups below
ten observations show their individual points instead of a density estimate.
Counts are in `considered_counts.csv` and `group_summary.csv`. No inferential tests
or claims of equivalence are made. PDF is vector output; PNG uses 300 dpi.


In [ ]:
S.apply()  # fonts, sizes and colours of the thesis
figure_rows = []
def save_figure(fig, name, caption):
    fig.savefig(FIGURE_DIR / f'{name}.pdf', bbox_inches='tight', facecolor='white')
    fig.savefig(FIGURE_DIR / f'{name}.png', dpi=300, bbox_inches='tight', facecolor='white')
    if SHOW_FIGURES and 'agg' != plt.get_backend().lower():
        plt.show()
    plt.close(fig)
    (FIGURE_DIR / f'{name}_caption.txt').write_text(caption, encoding='utf-8')
    figure_rows.append(dict(figure=name, caption=caption))

plot_metrics = [('deformation', 'Deformation, D'), ('area_um2', 'Projected area (µm²)'),
                ('speed_m_s', 'Speed (m/s)'), ('equivalent_diameter_um', 'Equivalent diameter (µm)'),
                ('axis_ratio', 'Major/minor axis ratio'), ('solidity', 'Solidity')]
for mode, data in cells_primary.groupby('imaging_mode', sort=True):
    types = sorted(data.cell_type.unique())
    flows = sorted(data.flow_rate_ul_min.unique())
    fig, axes = plt.subplots(2, 3, figsize=S.figsize('full', 0.72), layout='constrained')
    slot = .76 / len(types)
    for panel, (ax, (metric, ylabel)) in enumerate(zip(axes.flat, plot_metrics)):
        for j, flow in enumerate(flows):
            for k, ct in enumerate(types):
                pos = j + (k - (len(types)-1)/2) * slot
                v = data.loc[data.cell_type.eq(ct) & data.flow_rate_ul_min.eq(flow), metric].dropna().to_numpy(float)
                if not len(v):
                    continue
                color = COLORS.get(ct, '#666666')
                if len(v) >= 10 and np.ptp(v) > 0:
                    violin = ax.violinplot([v], positions=[pos], widths=.88*slot, showextrema=False, points=150)
                    for body in violin['bodies']:
                        body.set_facecolor(color); body.set_edgecolor(color); body.set_alpha(.3)
                else:
                    ax.scatter(pos + RNG.uniform(-.08*slot, .08*slot, len(v)), v, s=12, color=color, zorder=3)
                ax.boxplot([v], positions=[pos], widths=.26*slot, manage_ticks=False, showfliers=False,
                    patch_artist=True, boxprops=dict(facecolor=color, edgecolor=S.INK_2, linewidth=.8),
                    medianprops=dict(color=S.INK, linewidth=1.2), whiskerprops=dict(linewidth=.8), capprops=dict(linewidth=.8))
        ax.set_xticks(range(len(flows)), [f'{f:g}' for f in flows])
        ax.set(xlabel='Total flow rate (µL/min)', ylabel=ylabel, xlim=(-.55, len(flows)-.45))
        ax.yaxis.set_major_locator(MaxNLocator(5))
        S.ygrid(ax)
        ax.set_title(chr(65+panel), loc='left', fontweight='bold')
    fig.legend(handles=[Patch(facecolor=COLORS.get(ct, '#666666'), label=ct) for ct in types],
               loc='outside lower center', ncols=len(types))
    fig.suptitle(f'Track-level morphology and speed · {mode}')
    save_figure(fig, f'01_distributions_{mode}',
        f'Descriptive distributions in {mode}. Each retained track contributes its median. '
        'Violin widths are normalized separately and do not encode sample size. Boxes: IQR and median; '
        'whiskers: extrema within 1.5 IQR. All values enter densities and summaries. Counts are in group_summary.csv. '
        '600 µL/min excluded for chip failure. Acquisition dates differ between cell types.')
    fig, axes = plt.subplots(1, 3, figsize=S.figsize('full', 0.36), layout='constrained')
    for ax, (metric, label) in zip(axes, plot_metrics[:3]):
        for ct in types:
            part = data.loc[data.cell_type.eq(ct)].groupby('flow_rate_ul_min')[metric]
            s = part.agg(median='median', q25=lambda x: x.quantile(.25), q75=lambda x: x.quantile(.75))
            dx = (types.index(ct) - (len(types) - 1) / 2) * 0.03 * (max(flows) - min(flows) or 1)  # lateral offset per cell type
            ax.errorbar(s.index + dx, s['median'], yerr=[s['median']-s.q25, s.q75-s['median']],
                fmt='o', capsize=3, color=COLORS.get(ct, '#666666'), label=ct)
        ax.set(xlabel='Total flow rate (µL/min)', ylabel=label)
        ax.set_xticks(flows); S.ygrid(ax)
    axes[0].legend(frameon=False)
    fig.suptitle(f'Flow response · {mode}')
    save_figure(fig, f'02_flow_response_{mode}', 'Points: median of retained track medians; bars: between-track IQR, not confidence intervals. Optical modes are analyzed separately.')

fig, axes = plt.subplots(1, 2, figsize=S.figsize('full', 0.45), layout='constrained')
for ax, entity in zip(axes, ['detections', 'tracks']):
    stages = accounting.loc[accounting.entity.eq(entity)].groupby(['stage_order', 'stage']).remaining.sum()
    labels = [key[1].replace('_', '\n') for key in stages.index]
    ax.bar(range(len(stages)), stages.to_numpy(), color=S.NEUTRAL)
    ax.set_xticks(range(len(stages)), labels, rotation=30, ha='right', fontsize=8)
    ax.set_ylabel(f'Number of {entity}'); ax.set_title(f'Sequential retention: {entity}')
    ax.ticklabel_format(axis='y', style='plain')
    for i, value in enumerate(stages):
        ax.text(i, value, f'{value:,}', ha='center', va='bottom', fontsize=7)
save_figure(fig, '03_qc_accounting', 'Sequential counts in separate units. Each stage includes all previous criteria. Losses are additive within each panel; detection and track counts must not be added.')


## 8b. Single-detection figures

These figures use every selected detection (Section 7b). Violin widths are normalised separately. Groups with
fewer than ten detections show individual points. The area–deformation plot follows the representation used
in real-time deformability cytometry, because in the same channel larger cells experience higher stresses.

In [ ]:
def violin_panel(ax, data, metric, types, flows, slot):
    for j, flow in enumerate(flows):
        for k, ct in enumerate(types):
            pos = j + (k - (len(types) - 1) / 2) * slot
            v = data.loc[data.cell_type.eq(ct) & data.flow_rate_ul_min.eq(flow), metric].dropna().to_numpy(float)
            if not len(v):
                continue
            color = COLORS.get(ct, '#666666')
            if len(v) >= 10 and np.ptp(v) > 0:
                body = ax.violinplot([v], positions=[pos], widths=.88 * slot, showextrema=False, points=150)['bodies'][0]
                body.set_facecolor(color); body.set_edgecolor(color); body.set_alpha(.3)
            else:
                ax.scatter(pos + RNG.uniform(-.08 * slot, .08 * slot, len(v)), v, s=12, color=color, zorder=3)
            ax.boxplot([v], positions=[pos], widths=.26 * slot, manage_ticks=False, showfliers=False,
                       patch_artist=True, boxprops=dict(facecolor=color, edgecolor=S.INK_2, linewidth=.8),
                       medianprops=dict(color=S.INK, linewidth=1.2), whiskerprops=dict(linewidth=.8),
                       capprops=dict(linewidth=.8))
    ax.set_xticks(range(len(flows)), [f'{f:g}' for f in flows])
    ax.set(xlabel='Total flow rate (µL/min)', xlim=(-.55, len(flows) - .45))
    ax.yaxis.set_major_locator(MaxNLocator(5)); S.ygrid(ax)

detection_plot_metrics = [('deformation', 'Deformation, D'), ('deformation_rtdc', 'Deformation, 1 − c'),
                          ('axis_ratio', 'Major/minor axis ratio'), ('area_um2', 'Projected area (µm²)'),
                          ('confinement_ratio', 'Confinement ratio, d/H')]
if HAS_ORIENTATION:
    detection_plot_metrics.append(('inclination_deg', 'Inclination to flow (°)'))

for mode, data in selected.groupby('imaging_mode', sort=True):
    types = sorted(data.cell_type.unique())
    flows = sorted(data.flow_rate_ul_min.unique())
    slot = .76 / len(types)
    # --- distributions of single detections
    fig, axes = plt.subplots(2, 3, figsize=S.figsize('full', 0.72), layout='constrained')
    for panel, (ax, (metric, label)) in enumerate(zip(axes.flat, detection_plot_metrics)):
        violin_panel(ax, data, metric, types, flows, slot)
        ax.set_ylabel(label); ax.set_title(chr(65 + panel), loc='left', fontweight='bold')
    for ax in axes.flat[len(detection_plot_metrics):]:
        ax.set_visible(False)
    fig.legend(handles=[Patch(facecolor=COLORS.get(ct, '#666666'), label=ct) for ct in types],
               loc='outside lower center', ncols=len(types))
    fig.suptitle(f'Single-detection morphology · {mode}')
    save_figure(fig, f'06_detection_distributions_{mode}',
        f'Morphology of single detections in {mode}, including detections without a retained track. Each '
        'detection after detection QC, crop QC and removal of repeated observations contributes one value. '
        'D = (L − B)/(L + B) from the fitted ellipse, 1 − c from the circularity as in RT-DC, confinement ratio '
        '= equivalent diameter / channel height. Violin widths are normalised separately. Boxes: IQR and median. '
        'Counts are in detection_counts.csv. Acquisition dates differ between cell types.')

    # --- track level against detection level
    fig, axes = plt.subplots(1, 2, figsize=S.figsize('full', 0.36), layout='constrained')
    track_data = cells_primary.loc[cells_primary.imaging_mode.eq(mode)]
    span = (max(flows) - min(flows)) or 1
    for ax, (metric, label) in zip(axes, detection_plot_metrics[:2]):
        for k, ct in enumerate(types):
            for level, table, mfc, shift in [('tracks', track_data, None, -0.012), ('detections', data, 'white', 0.012)]:
                part = table.loc[table.cell_type.eq(ct)].groupby('flow_rate_ul_min')[metric]
                s = part.agg(median='median', q25=lambda x: x.quantile(.25), q75=lambda x: x.quantile(.75))
                if s.empty:
                    continue
                dx = ((k - (len(types) - 1) / 2) * 0.05 + shift) * span
                color = COLORS.get(ct, '#666666')
                ax.errorbar(s.index + dx, s['median'], yerr=[s['median'] - s.q25, s.q75 - s['median']], fmt='o',
                            capsize=3, color=color, mfc=mfc or color, label=f'{ct}, {level}')
        ax.set(xlabel='Total flow rate (µL/min)', ylabel=label); ax.set_xticks(flows); S.ygrid(ax)
    fig.legend(*axes[0].get_legend_handles_labels(), loc='outside upper center', ncols=2 * len(types))
    save_figure(fig, f'07_track_vs_detection_{mode}',
        f'{mode}: median and IQR of the deformation for retained tracks (filled, one median per track) and for '
        'all selected single detections (open). A difference between both indicates that the track requirement '
        'selects a subpopulation, for example slow cells at high flow rates.')

    # --- area against deformation, as in RT-DC
    fig, axes = plt.subplots(1, len(flows), figsize=S.figsize('full', 0.36), layout='constrained', sharey=True,
                             squeeze=False)
    y_metric = AREA_DEFORMATION_METRIC
    for ax, flow in zip(axes[0], flows):
        for ct in types:
            part = data.loc[data.cell_type.eq(ct) & data.flow_rate_ul_min.eq(flow), ['area_um2', y_metric]].dropna()
            if part.empty:
                continue
            color = COLORS.get(ct, '#666666')
            shown = part.sample(min(len(part), 3000), random_state=0)
            ax.scatter(shown.area_um2, shown[y_metric], s=2, alpha=.15, color=color, linewidths=0, rasterized=True)
            if len(part) >= 50:
                edges = np.unique(np.quantile(part.area_um2, np.linspace(0, 1, 9)))
                binned = part.groupby(pd.cut(part.area_um2, edges, include_lowest=True), observed=True)
                ax.plot(binned.area_um2.median(), binned[y_metric].median(), color=color, lw=1.6, marker='o', ms=3,
                        label=ct)
        ax.set(title=f'{flow:g} µL/min', xlabel='Projected area (µm²)'); S.ygrid(ax)
    axes[0, 0].set_ylabel(dict(detection_plot_metrics).get(y_metric, y_metric))
    handles = [plt.Line2D([], [], color=COLORS.get(ct, '#666666'), marker='o', lw=1.6, label=ct) for ct in types]
    fig.legend(handles=handles, loc='outside upper center', ncols=len(types))
    save_figure(fig, f'08_area_deformation_{mode}',
        f'{mode}: deformation against projected area for single detections (points, at most 3000 per cell type '
        'and panel) and medians in area octiles (lines, only for at least 50 detections). In the same channel, '
        'larger cells experience higher stresses, so deformation has to be compared at similar size.')

    # --- orientation of the elongation
    if HAS_ORIENTATION:
        fig, axes = plt.subplots(1, len(types) + 1, figsize=S.figsize('full', 0.34), layout='constrained')
        flow_colors = dict(zip(flows, S.levels(len(flows))))
        bins = np.linspace(0, 90, 19)
        for ax, ct in zip(axes, types):
            for flow in flows:
                v = data.loc[data.cell_type.eq(ct) & data.flow_rate_ul_min.eq(flow), 'inclination_deg'].dropna()
                if len(v) >= 10:
                    ax.hist(v, bins=bins, density=True, histtype='step', lw=1.4, color=flow_colors[flow],
                            label=f'{flow:g} µL/min')
            ax.axhline(1 / 90, color=S.NEUTRAL, lw=.8, ls=':')
            ax.set(title=ct, xlabel='Inclination to flow (°)', xticks=[0, 30, 60, 90]); S.ygrid(ax)
        axes[0].set_ylabel('Density')
        ax = axes[-1]
        for k, ct in enumerate(types):
            part = data.loc[data.cell_type.eq(ct)].groupby('flow_rate_ul_min').flow_aligned_deformation
            s = part.agg(median='median', q25=lambda x: x.quantile(.25), q75=lambda x: x.quantile(.75))
            dx = (k - (len(types) - 1) / 2) * 0.03 * ((max(flows) - min(flows)) or 1)
            ax.errorbar(s.index + dx, s['median'], yerr=[s['median'] - s.q25, s.q75 - s['median']], fmt='o', capsize=3,
                        color=COLORS.get(ct, '#666666'), label=ct)
        ax.axhline(0, color=S.NEUTRAL, lw=.8, ls=':')
        ax.set(xlabel='Total flow rate (µL/min)', ylabel='D · cos(2 · inclination)', xticks=flows); S.ygrid(ax)
        ax.legend(frameon=False)
        fig.legend(*axes[0].get_legend_handles_labels(), loc='outside upper center', ncols=len(flows))
        save_figure(fig, f'09_orientation_{mode}',
            f'{mode}: orientation of the major axis relative to the flow axis for single detections. Dotted line in '
            'the histograms: uniform distribution, expected for elongations without a preferred direction such as '
            'segmentation noise. Right: flow-aligned deformation D cos(2θ), positive for elongation along the flow. '
            'Points: median, bars: IQR.')

## 9. Representative individual trajectories and image sequences

One example per experiment, optical mode and flow is selected by proximity to the
arithmetic mean of retained track-median speeds. The search has no fixed distance
cutoff. Only retained identities are eligible. Images and measurements must still
be visually reviewed; a mean-like speed is not proof of a correct identity.
Plot intervals at their time midpoints. Accepted gaps use dashed trajectory
segments and hollow speed markers. No missing observations are interpolated,
and no temporal smoothing is applied.
Small galleries show the source detections, not newly generated segmentations.

In [ ]:
example_rows = []
for key, population in cells_primary.groupby(GROUPS, sort=True):
    target = population.speed_m_s.mean()
    ranked = population.assign(distance=(population.speed_m_s-target).abs()).sort_values(['distance', 'cell_uid'])
    chosen = None
    for row in ranked.itertuples():
        g = points.loc[points.cell_uid.eq(row.cell_uid)].sort_values('frame').copy()
        usable = g.loc[g.point_qc_ok]
        if len(usable) < MIN_TRACK_POINTS:
            continue
        chosen_indices = np.unique(np.linspace(0, len(usable)-1, 5).astype(int))
        samples = usable.iloc[chosen_indices]
        middle = usable.iloc[len(usable)//2]
        image_path = Path(middle.source_video_path)
        try:
            frames = {int(f): tifffile.imread(image_path, key=int(f)) for f in set(samples.frame) | {middle.frame}}
            if any(image.ndim != 2 for image in frames.values()):
                continue
        except (OSError, ValueError, IndexError):
            continue
        chosen = row
        break
    if chosen is None:
        continue
    row = chosen
    ct, flow, mode = row.cell_type, row.flow_rate_ul_min, row.imaging_mode
    color = COLORS.get(ct, '#666666')
    number = len(example_rows) + 1
    name = f'04_example_{number:02d}_{ct}_{flow:g}_{mode}'
    local_links = links.loc[links.cell_uid.eq(row.cell_uid)].sort_values('frame')
    t0 = g.time_s.min()
    background = frames[int(middle.frame)]
    # Shared linear contrast from the middle cell ROI, excluding the dark image margins.
    y0, y1 = max(0, int(middle.bbox_y0)-3), min(background.shape[0], int(middle.bbox_y1)+3)
    x0, x1 = max(0, int(middle.bbox_x0)-3), min(background.shape[1], int(middle.bbox_x1)+3)
    contrast_roi = background[y0:y1, x0:x1]
    lo, hi = np.percentile(contrast_roi if contrast_roi.size else background, [1,99])
    fig, axes = plt.subplots(1, 3, figsize=S.figsize('full', 0.34), layout='constrained')
    ax = axes[0]
    ax.imshow(background, cmap='gray', vmin=lo, vmax=max(hi, lo+1))
    for end in local_links.loc[local_links.used_in_primary].itertuples():
        pair = g.loc[g.detection_uid.isin([end.from_detection_uid, end.detection_uid])]
        ax.plot(pair.centroid_x_px, pair.centroid_y_px, color=color, linewidth=1.3,
                linestyle='--' if end.gap_accepted else '-')
    ax.scatter(usable.centroid_x_px.iloc[0], usable.centroid_y_px.iloc[0], marker='o', s=35, facecolors='white', edgecolors=color, label='Start')
    ax.scatter(usable.centroid_x_px.iloc[-1], usable.centroid_y_px.iloc[-1], marker='x', s=35, color=color, label='End')
    ax.scatter(middle.centroid_x_px, middle.centroid_y_px, s=100, facecolors='none', edgecolors=color)
    ax.set(xlabel='x (pixel)', ylabel='y (pixel)', title=f'A  Original frame {int(middle.frame)}')
    ax.set_xlim(-.5, background.shape[1]-.5); ax.set_ylim(background.shape[0]-.5, -.5)
    ax.legend(fontsize=7, loc='lower left')
    times = (local_links.segment_time_s-t0)*1000
    axes[1].plot(times, local_links.speed_m_s.where(local_links.used_in_primary), 'o-', ms=2.5, color=color, label='Total speed')
    axes[1].plot(times, local_links.axial_velocity_m_s.where(local_links.used_in_primary), '--', color=S.INK_2, lw=1, label='Axial velocity')
    bridged = local_links.gap_accepted & local_links.used_in_primary
    if bridged.any():
        axes[1].scatter(times.loc[bridged], local_links.loc[bridged, 'speed_m_s'],
                        s=36, facecolors='white', edgecolors=color, zorder=4, label='Accepted gap interval')
    axes[1].axhline(target, color=S.NEUTRAL, linestyle=':', label='Group mean of track medians')
    axes[1].set(xlabel='Elapsed time (ms)', ylabel='Speed / velocity (m/s)', title='B  Velocity')
    axes[1].legend(fontsize=7, frameon=False)
    axes[2].plot((g.time_s-t0)*1000, g.deformation.where(g.point_qc_ok), 'o-', ms=2.5, color=color)
    axes[2].set(xlabel='Elapsed time (ms)', ylabel='Deformation, D', title='C  Deformation')
    for ax in axes[1:]:
        S.ygrid(ax)
    fig.suptitle(f'{ct} · {flow:g} µL/min · {mode}')
    save_figure(fig, name, f'Illustrative original track {row.cell_uid}. Selected nearest to group mean speed, without a distance cutoff. Circle: start; cross: end; ring: cell in background frame. Invalid links break the lines. Accepted gaps are dashed in the trajectory and use hollow speed markers; speed averages over the actual elapsed interval. No interpolated observations or smoothing. Linear display contrast uses the 1st and 99th intensity percentiles of the middle cell ROI (3 pixel margin); limits are shared with the image sequence. Requires visual identity verification.')
    fig, axes = plt.subplots(1, len(samples), figsize=S.figsize('full', 0.26), squeeze=False, layout='constrained')
    half = int(np.ceil(max((usable.bbox_x1-usable.bbox_x0).max(), (usable.bbox_y1-usable.bbox_y0).max())/2 + 10))
    for ax, sample in zip(axes.flat, samples.itertuples()):
        image = frames[int(sample.frame)]
        ax.imshow(image, cmap='gray', vmin=lo, vmax=max(hi,lo+1))
        ax.add_patch(Rectangle((sample.bbox_x0-.5, sample.bbox_y0-.5), sample.bbox_x1-sample.bbox_x0,
                               sample.bbox_y1-sample.bbox_y0, fill=False, color=color, lw=1.1))
        ax.set_xlim(sample.centroid_x_px-half, sample.centroid_x_px+half)
        ax.set_ylim(sample.centroid_y_px+half, sample.centroid_y_px-half)
        ax.set_xticks([]); ax.set_yticks([])
        ax.set_title(f'{(sample.time_s-t0)*1000:.3f} ms\nFrame {int(sample.frame)}', fontsize=8)
    fig.suptitle(f'{ct} · {flow:g} µL/min · {mode} · identity review')
    save_figure(fig, name+'_frames', 'Up to five original frame excerpts, shared linear contrast from the middle cell ROI (1st/99th percentiles) and shared field size. Boxes are detection bounding boxes, not segmentation masks. The sequence is a visual aid, not proof of uninterrupted identity.')
    example_rows.append(dict(cell_uid=row.cell_uid, experiment_id=row.experiment_id, cell_type=ct,
        imaging_mode=mode, flow_rate_ul_min=flow, track_median_m_s=row.speed_m_s,
        group_mean_m_s=target, relative_distance=abs(row.speed_m_s-target)/target if target else np.nan,
        figure=name, n_bridged_gaps=int(bridged.sum()), visual_review='pending'))
examples = pd.DataFrame(example_rows)
example_coverage = considered.loc[~considered.condition_excluded, GROUPS].merge(examples[GROUPS] if len(examples) else pd.DataFrame(columns=GROUPS), on=GROUPS, how='left', indicator=True)
if example_coverage._merge.eq('left_only').any():
    print('No readable example for:', example_coverage.loc[example_coverage._merge.eq('left_only'), GROUPS].to_dict('records'))

## 10. Curation manifests, review sheets and reproducibility exports

The category manifests retain every detection ID and original path. The full
manifest includes source, crop, point and track reasons. Contact sheets are
stratified samples for review, not a claim that every image was visually read.
`accepted_candidate` means automatic checks passed; `review` stays out of the
ML candidate pool until a decision is entered. `accepted_reviewed` requires a
manual entry. Informational `crop_notes` do not affect eligibility. Originals
remain untouched.

Never randomly split neighboring frames into training and test sets. Use recording
groups, and preferably independent experiment days, to avoid leakage. Exact image
hashes and recording-group IDs are included. Multiple clips may originate from one
acquisition, so folder-level grouping is deliberately conservative. A single date
per cell type cannot establish generalization to independent days.
[Grouped evaluation reference](https://scikit-learn.org/stable/modules/cross_validation.html#cross-validation-iterators-for-grouped-data).

In [ ]:
review_samples = []
for category, part in crops.groupby('crop_category', sort=True):
    category_dir = CROP_DIR / category
    category_dir.mkdir(exist_ok=True)
    part.to_csv(category_dir / 'manifest.csv', index=False)
    samples = []
    for _, group in part.groupby(['cell_type', 'imaging_mode', 'flow_rate_ul_min'], dropna=False):
        samples.append(group.sample(min(4, len(group)), random_state=42))
    sample = pd.concat(samples, ignore_index=True) if samples else part.head(0)
    review_samples.append(sample.assign(review_sheet=category))
    if sample.empty:
        continue
    fig, axes = plt.subplots(int(np.ceil(len(sample)/6)), 6, figsize=(12, 2.1*np.ceil(len(sample)/6)), squeeze=False)
    for ax in axes.flat:
        ax.axis('off')
    for ax, row in zip(axes.flat, sample.itertuples()):
        try:
            im = tifffile.imread(row.crop_output_path)
            lo,hi = np.percentile(im,[1,99])
            ax.imshow(im, cmap='gray', vmin=lo, vmax=max(lo+1,hi))
        except (OSError, ValueError, IndexError, TypeError):
            ax.text(.5,.5,'Unavailable',ha='center',transform=ax.transAxes)
        reason = row.crop_hard_reasons or row.crop_review_reasons or 'automatic candidate'
        ax.set_title(f'{row.cell_type} {row.flow_rate_ul_min:g} {row.imaging_mode}\n{row.detection_uid[:10]}\n{reason[:38]}',fontsize=6)
    fig.suptitle(f'Crop review sample: {category}',fontsize=11)
    fig.tight_layout(rect=(0, 0, 1, 0.97))
    save_figure(fig, '05_crops_'+category, 'Deterministic stratified sample of crop decisions. Contrast scaled per crop for inspection; pixel values are not altered. Complete reasons and IDs are in crop manifests.')

all_tables = {
    'resolved_metadata': resolved_metadata, 'video_inventory': video_inventory,
    'detection_audit': points, 'link_audit': links,
    'links_retained': links.loc[links.used_in_primary], 'track_audit': track_qc,
    'cells_primary': cells_primary, 'group_summary': summary, 'sensitivity_summary': sensitivity,
    'sequential_accounting': accounting, 'exclusion_reason_counts': reason_counts,
    'qc_note_counts': qc_note_counts, 'gap_policy_counts': gap_policy_counts,
    'crop_policy_counts': crop_policy_counts,
    'detection_morphology': selected,
    'detection_summary': detection_summary, 'detection_counts': detection_counts,
    'track_repeatability': track_repeatability,
    'source_rejection_counts': source_rejections, 'considered_counts': considered,
    'unassigned_counts': unassigned_counts, 'crop_counts': crop_counts,
    'crop_manifest': crops, 'clip_source_mapping': clip_mapping, 'duplicate_track_pairs': duplicate_pairs, 'ml_representative_candidates': ml_representatives,
    'example_tracks': examples, 'figure_captions': pd.DataFrame(figure_rows),
    'crop_review_samples': pd.concat(review_samples, ignore_index=True) if review_samples else pd.DataFrame(),
}
for name, table in all_tables.items():
    # Compress only large audit tables; small tables remain easy to open in Excel.
    suffix = '.csv.gz' if len(table) > 100000 else '.csv'
    table.to_csv(TABLE_DIR / (name + suffix), index=False)
    print('Exported', name, len(table), flush=True)

configs = {key: globals()[key] for key in ['PIPELINE_VERSION', 'EXPERIMENTS', 'PBS_TEMPERATURE_C',
    'PBS_DENSITY_KG_M3', 'FLOW_DIRECTION_DEG', 'EXCLUDED_FLOW_RATES', 'EDGE_MARGIN_PX',
    'MIN_TRACK_POINTS', 'MIN_VALID_SEGMENTS', 'MIN_SHARED_OBSERVATIONS', 'REVERSE_TOLERANCE_PX', 'MAX_TRACK_AREA_CV',
    'AREA_SPIKE_RATIO', 'ROBUST_Z_LIMIT', 'MAX_LINK_DISTANCE_PX', 'MAX_SPEED_M_S',
    'MAX_MISSING_SAMPLED_DETECTIONS', 'GAP_SPEED_RATIO_MIN', 'GAP_SPEED_RATIO_MAX',
    'GAP_MAX_DIRECTION_CHANGE_DEG', 'BBOX_OVERLAP_NOTE_THRESHOLD',
    'DETECTION_EXCLUDE_UNSTABLE_AREA_TRACKS', 'AREA_DEFORMATION_METRIC', 'EXPOSURE_US']}
versions = {}
for package in ['numpy','pandas','matplotlib','tifffile']:
    versions[package] = version(package)
provenance = dict(created_utc=datetime.now(timezone.utc).isoformat(), configuration=configs,
    source_manifest=source_manifest, event_sources=event_sources, metadata_sha256=file_hash(METADATA_CSV) if METADATA_CSV.exists() else None,
    manual_review_sha256=file_hash(REVIEW_CSV), package_versions=versions,
    n_detections=len(points), n_tracks_primary=len(cells_primary), n_single_detections=len(selected), n_links_primary=int(links.used_in_primary.sum()),
    n_crops=len(crops), n_crop_candidates=int(crops.ml_eligible.sum()),
    n_ml_representatives=len(ml_representatives),
    n_accepted_gap_links=int(links.gap_accepted.sum()),
    n_primary_tracks_with_gaps=int((track_qc.track_retained & track_qc.n_bridged_gaps.gt(0)).sum()),
    n_bbox_only_review_resolved=int(bbox_only_candidates.sum()), n_visually_reviewed=int(crops.manual_decision.ne('').sum()))
(RUN_DIR / 'provenance.json').write_text(json.dumps(provenance, indent=2, default=str), encoding='utf-8')

# A small local report links figures and CSVs without copying the raw images.
counts_html = considered.to_html(index=False, float_format=lambda x: f'{x:g}')
parts = ['<!doctype html><meta charset="utf-8"><title>Cell analysis audit</title>',
    '<style>body{font:15px Arial;max-width:1200px;margin:30px auto}table{border-collapse:collapse}td,th{padding:6px;border:1px solid #ddd}img{max-width:100%}</style>',
    '<h1>Cell analysis and curation audit</h1>',
    '<p>Descriptive track-level analysis. 600 uL/min excluded for chip failure. Original images and databases unchanged.</p>',
    '<p>Bounding-box overlap alone is informational. One missing sampled detection may be bridged only with consistent ordinary links on both sides. The audit and sensitivity tables record the effect.</p>',
    '<h2>Considered data</h2>', counts_html,
    '<p>Tracks are not necessarily unique biological cells. Crop candidates have not all been visually validated.</p>',
    '<h2>Tables</h2><ul>']
for path in sorted(TABLE_DIR.iterdir()):
    parts.append(f'<li><a href="tables/{html.escape(path.name)}">{html.escape(path.name)}</a></li>')
parts.append('</ul><h2>Figures</h2>')
for row in figure_rows:
    name = row['figure']
    parts.append(f'<h3>{html.escape(name)}</h3><a href="figures/{name}.pdf">PDF</a><img loading="lazy" src="figures/{name}.png"><p>{html.escape(row["caption"])}</p>')
(RUN_DIR / 'report.html').write_text('\n'.join(parts), encoding='utf-8')
print('Completed. Open:', RUN_DIR / 'report.html')

## 11. Optional physical crop export (disabled by default)

Manifests are the fastest and smallest organization and already separate the
categories. If an ML loader needs directories, copy the chosen manifest below.
Source TIFF bytes remain unchanged. This copies only one candidate per retained
track by default, not every near-identical frame. Never edit the source images.


In [ ]:
COPY_REPRESENTATIVE_CROPS = True
if COPY_REPRESENTATIVE_CROPS:
    import shutil
    copied = []
    for row in ml_representatives.itertuples():
        destination = CROP_DIR / 'representative_copies' / row.cell_type / row.imaging_mode / (row.detection_uid + '.tif')
        destination.parent.mkdir(parents=True, exist_ok=True)
        if not destination.exists():
            shutil.copy2(row.crop_output_path, destination)
        copied.append(dict(detection_uid=row.detection_uid, original=row.crop_output_path, copy=str(destination)))
    pd.DataFrame(copied).to_csv(CROP_DIR / 'representative_copy_manifest.csv', index=False)
    print('Copied representative crops:', len(copied))
else:
    print('Crop sorting uses manifests; all original images remain at their original paths.')


## Interpretation checklist

- Review the resolved calibration and flow sources before interpreting SI values.
- Report track counts, not a presumed number of unique biological cells.
- Report sequential losses and overlapping reason counts with their correct units.
- Inspect crop and example-track galleries; automatic flags are not ground truth.
- Compare primary results with the area-CV, crop-QC and no-bridged-gap sensitivity summaries.
- Keep BF and Ph2 separate unless pooling is experimentally justified.
- Do not infer mechanical stiffness or local forces from cell speed alone.
- Use independent acquisitions for ML validation; track/frame random splits leak identity.
- Document PBS temperature/density as assumptions unless independently measured.
- The exported figure captions and audit report are part of the reproducible result.
- Report single-detection results in detections, not cells; unassigned detections can repeat a cell.
- Compare track-level and detection-level deformation before interpreting high flow rates.
- Read single-frame deformation together with the within-track repeatability and the orientation.